# colab_15 — Geneformer CPT evals #1 & #2, **per-study regime** (3 checkpoints)

Closes the gap colab_14 left open: its three per-study LoRA checkpoints (SEA-AD / Li2025 / Haney2024) are established as *real and mutually divergent* on detector #1, but that is a drift-only finding — no eval has ever been run on them. This notebook runs the same pre-committed battery colab_12 ran on the aggregated checkpoint — **eval #1** (substate linear probe) and **eval #2** (APOE-carrier recovery, Stanton core) — across all three per-study checkpoints, with the aggregated checkpoint and the zero-shot baseline included as reference columns so the regimes are compared on identical cells.

**Two tiers, because the extraction points cost different amounts.**

- **Tier 1 (`EXTRACT_L0=False`, no GPU, no compute units)** — evals at `emb_layer=-1` over the full 142,588-cell substrate. Every embedding it needs already exists on Drive (colab_09 zero-shot, colab_11 aggregated v2, colab_14 per-study), so this tier is pure CPU work.
- **Tier 2 (`EXTRACT_L0=True`, GPU)** — the locked dual-extraction design (Option 3): the same evals at **both** `emb_layer=-1` and `emb_layer=0`, so the −1→L0 delta gives the eval-space read of layer 11's absorbed share. The three per-study L0 embeddings do not exist and must be produced. To keep that affordable this tier runs on a donor-stratified subsample (cap `SUB_CAP` cells per donor x lineage x substate) rather than all 142,588 cells; the zero-shot and aggregated L0 matrices colab_12 already produced are subset to the same cells, so all five variants are compared on one population.

**Why subsampling tier 2 costs little power.** These evals are donor-held-out by construction: the probe trains on train-split donors and is scored on the 22 disjoint test donors. The effective sample size is therefore the *donor* count, not the cell count — cells within a donor are strongly correlated. Capping per donor x lineage x substate keeps all 145 donors and all substate classes present, and `apoe_carrier` is a donor-level property so eval #2's class composition is preserved at donor granularity.

**What is being asked.** colab_12 found eval #1 and eval #2 both null for the aggregated checkpoint at both extraction points. Two things could still show up here: (a) a per-study checkpoint could move an axis the aggregated one did not — the regimes differ, so a null in one is not automatically a null in the other; (b) colab_14 found the three checkpoints' population-matched `drift_all` spans 1.81x (SEA-AD 0.00543 / Haney2024 0.00359 / Li2025 0.00300), which gives an ordering prediction to check against the eval deltas. Both readings are reported; neither is a hypothesis test at three checkpoints.

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the environment, set the run flags

Two live switches. `SMOKE` is the usual plumbing toggle. `EXTRACT_L0` is the tier switch and the only thing in this notebook that costs GPU time: with it off, nothing here touches a GPU and Geneformer itself is not installed, because every embedding tier 1 reads already exists on Drive. Both are committed `False`; flip them in the notebook on Colab. This notebook's runtime metadata does not request a GPU, so Colab will default to a CPU runtime — matching tier 1's zero-cost design. If running tier 2 (`EXTRACT_L0=True`), manually switch Runtime > Change runtime type > GPU before executing this cell.

In [1]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

!pip install -r {REPO_PATH}/requirements_geneformer.txt

# ------------------------------ the two live switches ------------------------------
SMOKE      = False   # plumbing run on a tiny subsample; never writes the audit trace
EXTRACT_L0 = False   # False -> tier 1 only (L-1 from existing embeddings; CPU, no compute units)
                     # True  -> also build the 3 per-study L0 embeddings (GPU, 3 extraction passes)
# ----------------------------------------------------------------------------------

SMOKE_N_PER_GROUP = 40      # cells per (lineage x split x substate) group when SMOKE
SUB_CAP = 40                # tier-2 subsample: cells per (donor x lineage x substate)
SEED    = 0
SUFFIX  = "_SMOKE" if SMOKE else ""

from datetime import date
TODAY   = date.today().isoformat()
STUDIES = ["SEA-AD", "Li2025", "Haney2024"]
SLUG    = {"SEA-AD": "seaad", "Li2025": "li2025", "Haney2024": "haney2024"}

# Geneformer and a GPU are needed ONLY for tier 2's L0 extraction passes. Tier 1 re-reads
# embeddings that are already on Drive, so it runs on a CPU runtime at zero compute cost.
if EXTRACT_L0:
    GENEFORMER_REPO = "/content/Geneformer"
    if not os.path.exists(GENEFORMER_REPO):
        !git lfs install
        !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
    # Pin the clone. An unpinned HEAD is a silent input change to every embedding produced below,
    # and these must be comparable to L-1 embeddings made months earlier (colab_13 fix 0919506).
    GENEFORMER_PIN = "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5"
    subprocess.run(["git", "-C", GENEFORMER_REPO, "checkout", "-q", GENEFORMER_PIN], check=True)
    !cd {GENEFORMER_REPO} && pip install .
    # torchao (Colab-preinstalled, unused) hard-raises inside peft dispatch below its floor.
    !pip uninstall -y torchao -q
    GENEFORMER_COMMIT = subprocess.run(["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
                                       capture_output=True, text=True).stdout.strip()
    assert GENEFORMER_COMMIT == GENEFORMER_PIN, \
        f"Geneformer HEAD {GENEFORMER_COMMIT} != pin {GENEFORMER_PIN}"
    import torch
    assert torch.cuda.is_available(), "EXTRACT_L0=True needs a GPU runtime -- select one and re-run"
    print("Geneformer commit:", GENEFORMER_COMMIT, "| GPU:", torch.cuda.get_device_name(0))
else:
    # Tier 1 never installs Geneformer, but scanpy/anndata/scikit-learn are pulled in by
    # Geneformer's OWN `pip install .` in every other notebook (its setup.py depends on them) --
    # requirements_geneformer.txt deliberately does not list them (see that file's header). Without
    # Geneformer installed here, 2a's `import scanpy` would crash on a stock Colab image.
    !pip install -q scanpy anndata
    # filled in at 4a from the commit the existing embeddings were actually recorded under,
    # rather than asserted here from a value this run cannot verify.
    GENEFORMER_COMMIT = None
    print("EXTRACT_L0=False -- tier 1 only (L-1 from existing embeddings). No GPU, no Geneformer install.")

print(f"Python {sys.version.split()[0]} | SMOKE={SMOKE} | EXTRACT_L0={EXTRACT_L0} | suffix='{SUFFIX}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EXTRACT_L0=False -- tier 1 only (L-1 from existing embeddings). No GPU, no Geneformer install.
Python 3.12.13 | SMOKE=False | EXTRACT_L0=False | suffix=''


> **Interpretation — environment installed, tier-1 branch confirmed (1a).** Every package pip touched here was already present in this runtime (all lines read "Requirement already satisfied") — psutil, transformers 4.57.0, peft 0.19.1, datasets 5.0.0, and their full dependency tree including torch 2.11.0+cu128 and the CUDA 12.8 toolkit. The printed banner "EXTRACT_L0=False -- tier 1 only (L-1 from existing embeddings). No GPU, no Geneformer install." confirms the structural branch this run took: the Geneformer git-clone-install and commit-pin/HEAD-assert steps, which only exist inside the `if EXTRACT_L0:` block, never executed. The closing line (`Python 3.12.13 | SMOKE=False | EXTRACT_L0=False | suffix=''`) is the run's own confirmation of its two live switches: this is a real, full-scale run rather than a plumbing dry run, and it is scoped to tier 1 only.

### 1b — pip freeze + env JSON (records the exact eval-run stack)

Tier 2 re-merges LoRA adapters with `peft`, whose merge semantics are already flagged correctness-critical in `docs/ASSUMPTIONS.md`, so the exact stack matters. Recorded on tier-1 runs too, where the sklearn/scanpy versions govern every number the notebook produces.

In [2]:
import json, platform, subprocess, sys

NOTEBOOK_ID = "colab_15"
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "smoke":          SMOKE,
    "extract_l0":     EXTRACT_L0,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "geneformer_commit":    GENEFORMER_COMMIT,   # None on a tier-1 run: Geneformer is not installed
    "scanpy_version":       _ver("scanpy"),
    "anndata_version":      _ver("anndata"),
    "sklearn_version":      _ver("sklearn"),
    "torch_version":        _ver("torch"),
    "transformers_version": _ver("transformers"),
    "peft_version":         _ver("peft"),
    "datasets_version":     _ver("datasets"),
    "geneformer_version":   _ver("geneformer"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

/tmp/ipykernel_8205/1044109458.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  return __import__(mod).__version__
/tmp/ipykernel_8205/1044109458.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  return __import__(mod).__version__


{
  "notebook_id": "colab_15",
  "date": "2026-07-26",
  "smoke": false,
  "extract_l0": false,
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "os_release": "6.6.122+",
  "gpu": "GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-9e03436a-3638-c42a-925e-f3c8577abdf7)",
  "cuda": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "git_commit": "81fac2e850f6f2597f14c3cff81dfdaccbabb4bf",
  "geneformer_commit": null,
  "scanpy_version": "1.12.3",
  "anndata_version": "0.13.2",
  "sklearn_version": "1.6.1",
  "torch_version": "2.11.0+cu128",
  "transformers_version": "4.57.0",
  "peft_version": "0.19.1",
  "datasets_version": "5.0.0",
  "geneformer_version": null
}


> **Interpretation — env snapshot recorded, and a real gap worth flagging (1b).** The FutureWarnings are cosmetic — a deprecated way of reading a package's version string, not a functional problem — and the snapshot itself is what matters: `git_commit` is `81fac2e...`, the exact commit this run was pulled from; `geneformer_commit` and `geneformer_version` are both `null` at this point in the run, which is the correct signature of a tier-1 run so far (nothing has installed the Geneformer package yet, so there is nothing to record) rather than a missing-value bug. `geneformer_commit` does not stay null, though: cell 4a later backfills it from the recorded audit trail and rewrites this same JSON file in place, so the version on disk after the full run reads the real commit (`04c2b2e8...`) with a `geneformer_commit_source` note explaining it was recovered from history rather than installed live. Only `geneformer_version` has no such backfill and stays `null` permanently, since the installed Geneformer package doesn't expose a version string even when it is installed. The pinned library versions that actually govern every downstream number are `scanpy 1.12.3`, `anndata 0.13.2`, and `sklearn 1.6.1` — since tier 1 does no model training or extraction, this is effectively an sklearn-and-scanpy run, and `sklearn 1.6.1` in particular is the version whose `LogisticRegression`/`KNeighborsClassifier`/`silhouette_score` behavior every probe and k-NN result below depends on. One thing worth flagging plainly: the snapshot also records `"gpu": "NVIDIA A100-SXM4-80GB"` (and the notebook's own saved metadata separately stamps `accelerator: GPU`) — this session was connected to a GPU-backed runtime even though tier 1's entire cost argument (zero compute units) assumes a plain CPU runtime, since nothing in this branch ever calls the GPU. Being connected to an A100 spends compute-unit budget for the connection itself, independent of whether the code uses it; a future tier-1-only run should explicitly pick a no-accelerator runtime to actually realize the zero-cost design.

## 2 — Load the substrate and validate the schema

### 2a — Rebuild the glia substrate (deterministic; same `cell_index` as every saved embedding)

Identical to colab_12/13/14: concatenate the labelled microglia and astrocyte subsets in the same order and re-mint `cell_index` as a positional counter. Every embedding file this notebook loads carries the `cell_index` it was written under, so this rebuild is what makes row *i* the same cell in all of them. The 142,588 assert is the hard stop that the upstream subset files have not changed underneath.

In [3]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"
micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "region", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("combined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("substate:", glia.obs["substate"].value_counts(dropna=False).to_dict())
print("apoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())
print("study_id:", glia.obs["study_id"].value_counts().to_dict())

assert glia.n_obs == 142588, f"expected 142,588-cell substrate, got {glia.n_obs} -- subset files changed"

combined glia: (142588, 26514)
lineage: {'astrocyte': 87783, 'microglia': 54805}
substate: {'resting': 48147, 'reactive': 28465, 'intermediate': 28022, 'homeostatic': 25845, 'activated': 12109}
apoe_carrier: {'noncarrier': 70169, 'carrier': 49831, 'e2': 22588}
study_id: {'SEA-AD': 75634, 'Li2025': 46870, 'Haney2024': 20084}


> **Interpretation — substrate rebuilt, all cells accounted for (2a).** The rebuilt glia object is 142,588 cells by 26,514 genes, split 87,783 astrocyte / 54,805 microglia — an exact match to the frozen substrate every foundation-model notebook in this project has used since colab_09. The `substate` dict combines both lineages' independent labels into one column: astrocyte contributes resting (48,147) and reactive (28,465), microglia contributes homeostatic (25,845) and activated (12,109), and the two lineages' separate "intermediate" buckets are summed together (16,851 micro + 11,171 astro = 28,022) since intermediate cells are excluded from the binary probes run later. All five substate numbers sum to 142,588 — nothing was silently dropped. `apoe_carrier` splits noncarrier 70,169 / carrier 49,831 / e2 22,588 (also summing to 142,588); `e2` is deliberately kept as its own bucket rather than folded into "noncarrier," since E2-without-E4 donors are excluded from the primary binary carrier axis by design, not treated as noncarriers. `study_id` gives SEA-AD 75,634 / Li2025 46,870 / Haney2024 20,084, also summing exactly. The one structural difference from colab_12/14: this cell carries no raw-counts guard of its own — that check moved to cell 4c, the one place downstream where it actually matters (tokenization, the first step that reads the counts as numbers rather than just as labels).

### 2b — Fail loud on the obs schema the evals depend on

`region` is required here (unlike colab_14): eval #2's confound audit reports held-out carriers by study x region. Known caveat carried from colab_12 — Haney2024's `region` is the literal string `"unknown"` for all its cells, so that study collapses into one uninformative region bucket. Non-null, so the assert legitimately passes; it just limits what the region axis of the audit can show.

Unlike colab_14 there is no raw-counts guard, because tier 1 never tokenizes. It is applied in 4c, where it is actually load-bearing.

In [4]:
REQUIRED_OBS = ["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "region", "donor_id"]
missing_cols = [c for c in REQUIRED_OBS if c not in glia.obs.columns]
assert not missing_cols, (
    f"substrate missing required obs columns: {missing_cols} "
    "(region is inherited from the colab_07/08 subsets -- check they carry it)")

for col in REQUIRED_OBS:
    n_null = int(pd.isna(glia.obs[col]).sum())
    assert n_null == 0, f"{col} has {n_null} null values -- the eval audits require it complete"

assert set(glia.obs["lineage"]) == {"microglia", "astrocyte"}, "unexpected lineage values"
assert set(glia.obs["substate"]) <= {"homeostatic", "activated", "resting", "reactive", "intermediate"}, \
    f"unexpected substate values: {set(glia.obs['substate'])}"
assert set(glia.obs["apoe_carrier"]) <= {"carrier", "noncarrier", "e2"}, \
    f"unexpected apoe_carrier values: {set(glia.obs['apoe_carrier'])}"
assert set(glia.obs["study_id"]) == set(STUDIES), \
    f"substrate studies {set(glia.obs['study_id'])} != the 3 per-study checkpoints {set(STUDIES)}"
print("obs schema OK | region x study:")
print(pd.crosstab(glia.obs["region"], glia.obs["study_id"]))

obs schema OK | region x study:
study_id         Li2025  SEA-AD  Haney2024
region                                    
MTG                   0   75634          0
temporal cortex   46870       0          0
unknown               0       0      20084


> **Interpretation — schema passed, and the Haney "unknown"-region caveat carries forward (2b).** The obs schema assert passed silently, meaning every column the evals depend on (`cell_index`, `lineage`, `substate`, `apoe_carrier`, `study_id`, `region`) is present with zero nulls across all 142,588 cells, and `study_id` holds exactly the three expected study names the per-study loops key off of. The printed region × study crosstab shows each study occupies exactly one region value: SEA-AD → "MTG" (75,634 cells), Li2025 → "temporal cortex" (46,870 cells), and Haney2024 → the literal placeholder string "unknown" (20,084 cells) — a known limitation carried over from colab_12, not a new defect. Because "unknown" is a non-null string, the schema assert legitimately passes; it just means every table downstream that breaks results down by region (notably 7a's confound audit) collapses all of Haney2024 into one uninformative bucket rather than a real anatomical label.

## 3 — Held-out split verification

### 3a — Rebuild the frozen donor split and hard-stop on any mismatch

The standing split-verification rule. Re-derives the donor split from the committed metadata and hard-asserts it against the frozen reference on four independent axes — seed 32, margin 10, donor counts 101/22/22, cell counts 94963/23824/23801 — plus the per-donor identity map in `outputs/donor_split.json`, which is the only check that catches a metadata edit that leaves all the aggregate numbers intact while silently moving one donor between splits. A mismatch is a hard stop, never a new split: the per-study checkpoints being evaluated were trained under this exact split, so a different one would make every delta below meaningless.

In [5]:
import json
from sklearn.model_selection import train_test_split

DONOR_META_PATH = os.path.join(REPO_PATH, "outputs", "donor_metadata.csv")
SPLIT_FRACS     = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED_POOL       = range(200)

assert os.path.exists(DONOR_META_PATH), f"need committed donor metadata {DONOR_META_PATH} (colab_11 minted it)"
donor_meta = pd.read_csv(DONOR_META_PATH, dtype=str)
sub_donors = set(glia.obs["donor_id"].astype(str))
donor_meta = donor_meta[donor_meta["donor_id"].isin(sub_donors)].reset_index(drop=True)
assert donor_meta["donor_id"].is_unique, "a donor_id maps to >1 metadata row"

def _study_split(seed):
    d_tr, d_te = train_test_split(donor_meta["donor_id"], test_size=SPLIT_FRACS["test"],
                                  random_state=seed, stratify=donor_meta["study_id"])
    strat_tr = donor_meta.set_index("donor_id").loc[d_tr.values, "study_id"]
    d_tr, d_va = train_test_split(d_tr, test_size=SPLIT_FRACS["val"] / (SPLIT_FRACS["train"] + SPLIT_FRACS["val"]),
                                  random_state=seed, stratify=strat_tr)
    return set(d_tr), set(d_va), set(d_te)

def _test_margin(d_test):
    m = donor_meta[donor_meta["donor_id"].isin(d_test)]
    return int(min((m["apoe_carrier"] == "carrier").sum(), (m["apoe_carrier"] == "noncarrier").sum(),
                   (m["diagnosis"] == "AD").sum(), (m["diagnosis"] == "Control").sum()))

best_seed = max(SEED_POOL, key=lambda s: _test_margin(_study_split(s)[2]))
d_train, d_val, d_test = _study_split(best_seed)
margin = _test_margin(d_test)
split_map = {**{d: "train" for d in d_train}, **{d: "val" for d in d_val}, **{d: "test" for d in d_test}}

# HARD STOP against the committed donor-identity map, not just the aggregate counts below --
# a metadata edit that leaves seed/margin/donor-counts unchanged could otherwise silently
# reassign a specific donor's split and corrupt every delta downstream.
SPLIT_REF_PATH = os.path.join(REPO_PATH, "outputs", "donor_split.json")
assert os.path.exists(SPLIT_REF_PATH), f"need committed split reference {SPLIT_REF_PATH} (colab_11 minted it)"
with open(SPLIT_REF_PATH) as f:
    split_ref = json.load(f)["donor_split"]
assert set(split_map) == set(split_ref), (
    f"donor set differs from the committed reference: {set(split_map) ^ set(split_ref)} not shared")
mismatched = {d: (split_map[d], split_ref[d]) for d in split_ref if split_map[d] != split_ref[d]}
assert not mismatched, (
    f"{len(mismatched)} donor(s) assigned a different split than the committed reference: "
    f"{dict(list(mismatched.items())[:5])}")
print(f"split_map verified identical to {os.path.relpath(SPLIT_REF_PATH, REPO_PATH)} "
      f"for all {len(split_ref)} donors")

glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map).astype("category")
assert not glia.obs["split"].isna().any(), "some cells' donor received no split assignment"

n_donors = {k: int(sum(1 for v in split_map.values() if v == k)) for k in ("train", "val", "test")}
n_cells  = glia.obs["split"].value_counts().to_dict()
test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True).round(3).to_dict()
print(f"seed {best_seed} | margin {margin} | donors {n_donors} | cells {n_cells}")
print("test study fractions:", test_by_study)

# HARD STOP: match the frozen reference exactly (standing split-verification rule)
assert best_seed == 32,  f"seed {best_seed} != reference 32 -- split drifted, do NOT proceed"
assert margin == 10,     f"margin {margin} != reference 10 -- split drifted"
assert n_donors == {"train": 101, "val": 22, "test": 22}, f"donor counts {n_donors} != reference"
assert n_cells.get("train") == 94963 and n_cells.get("val") == 23824 and n_cells.get("test") == 23801, \
    f"cell counts {n_cells} != reference 94963/23824/23801"
print("split verification PASSED -- matches the frozen reference")

# --- SMOKE subsample: AFTER the split is assigned, BEFORE any tokenization (the dominant cost) ---
if SMOKE:
    keep = (glia.obs.groupby(["lineage", "split", "substate"], observed=True)
            .apply(lambda g: g.sample(min(len(g), SMOKE_N_PER_GROUP), random_state=SEED))
            .index.get_level_values(-1))
    glia = glia[glia.obs.index.isin(keep)].copy()
    print(f"[SMOKE] subsampled to {glia.n_obs} cells | split:", glia.obs["split"].value_counts().to_dict(),
          "| by study:", glia.obs["study_id"].value_counts().to_dict())

split_map verified identical to outputs/donor_split.json for all 145 donors
seed 32 | margin 10 | donors {'train': 101, 'val': 22, 'test': 22} | cells {'train': 94963, 'val': 23824, 'test': 23801}
test study fractions: {'SEA-AD': 0.567, 'Li2025': 0.3, 'Haney2024': 0.133}
split verification PASSED -- matches the frozen reference


> **Interpretation — split verification passed on all five checks (3a).** The donor split is not drawn fresh — it is a deterministic function of the same donor metadata and stratified-search procedure used to build the split every CPT training run in this project was trained under, so re-deriving it here from the unchanged substrate must reproduce the identical result. It did, on all five independent axes checked: seed 32, worst-case test-donor margin 10, donor counts 101/22/22 (train/val/test), cell counts 94,963/23,824/23,801, and — the check that matters most here — the per-donor identity map matched `outputs/donor_split.json` exactly for all 145 donors. The first four are aggregate numbers that could in principle stay identical even if one specific donor quietly swapped sides between runs; only the identity check catches that failure mode. This particular check matters more for this notebook than almost any other in the project, because the three per-study checkpoints being evaluated here were trained under exactly this split — a silently different split would not produce an obviously-wrong number, it would produce a plausible-looking one computed partly on cells a checkpoint had already seen during its own training.

## 4 — Embedding inventory, tier-2 subsample, and the L0 extraction passes

### 4a — Inventory the embeddings from the recorded audit trail

The per-study embedding and adapter paths are **read out of `outputs/audit_report.json`**, not reconstructed from a filename convention: colab_14 recorded where it wrote them, so reading that record is what makes this notebook point at the real checkpoints rather than at a path that merely looks right. The same entry supplies each checkpoint's `drift_all` and training-step count, which §8 needs.

Three guards worth naming. All five embedding sets must carry the *same* Geneformer commit, or the rank encodings are not comparable and no delta between them means anything. Every tier-1 L-1 file must exist before anything else runs. And on a tier-2 run, colab_12's zero-shot and aggregated L0 files must already be present — those two are re-used rather than rebuilt, which is most of why tier 2 is affordable.

In [6]:
import json

AUDIT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_PATH) as f:
    audit = json.load(f)

GF_DIR = os.path.join(DRIVE_ROOT, "geneformer")

# --- colab_14's per-study record: the authority on where the 3 checkpoints live ---
ps = audit["geneformer_cpt_per_study"]
assert ps["status"] == "computed", "colab_14's per-study run is not recorded as computed"
assert ps["n_cells"] == 142588, f"colab_14 ran on {ps['n_cells']} cells, not the 142,588 substrate"
assert set(ps["per_study"]) == set(STUDIES), f"unexpected studies in the record: {set(ps['per_study'])}"

# --- the aggregated reference (colab_11 v2) and the zero-shot baseline (colab_09) ---
agg = audit["geneformer_cpt_aggregated_v2"]
zs  = audit["geneformer_zeroshot"]
assert agg["status"] == "computed", "colab_11's aggregated CPT run is not recorded as computed"
assert zs["status"] == "computed", "colab_09's zero-shot run is not recorded as computed"

# Every embedding compared below must come from ONE Geneformer commit: a different commit means a
# different rank encoding, which would make each delta a mix of model change and tokenizer change.
_commits = {ps["geneformer_commit"], agg["geneformer_commit"], zs["geneformer_commit"]}
assert len(_commits) == 1, f"the embeddings span multiple Geneformer commits: {_commits}"
EMB_COMMIT = _commits.pop()
if GENEFORMER_COMMIT is None:
    # tier-1 run: Geneformer is not installed, so adopt the commit the embeddings were recorded
    # under and rewrite the env JSON, which 1b wrote as null before this was knowable.
    GENEFORMER_COMMIT = EMB_COMMIT
    env_snapshot["geneformer_commit"] = EMB_COMMIT
    env_snapshot["geneformer_commit_source"] = "audit_report.json (Geneformer not installed this run)"
    with open(ENV_JSON_PATH, "w") as f:
        json.dump(env_snapshot, f, indent=2)
else:
    assert GENEFORMER_COMMIT == EMB_COMMIT, (
        f"installed Geneformer {GENEFORMER_COMMIT} != the commit that produced the existing "
        f"embeddings {EMB_COMMIT} -- new L0 embeddings would not be comparable to the L-1 set")
print("embedding Geneformer commit:", EMB_COMMIT)

MODELS   = ["zeroshot", "aggregated"] + [SLUG[s] for s in STUDIES]
ADAPTERS = {SLUG[s]: os.path.join(DRIVE_ROOT, ps["per_study"][s]["adapter_file"]) for s in STUDIES}

DRIFT_ALL = {SLUG[s]: ps["per_study"][s]["detector_1"]["drift_all"] for s in STUDIES}
DRIFT_ALL["aggregated"] = agg["detector_1"]["drift_all"]
STEPS = {SLUG[s]: ps["per_study"][s]["max_steps"] for s in STUDIES}
STEPS["aggregated"] = agg["train"]["max_steps"]
N_TRAIN = {SLUG[s]: ps["per_study"][s]["n_train"] for s in STUDIES}
N_TRAIN["aggregated"] = agg["n_train_cells"]

L1_PATHS = {
    "zeroshot":   os.path.join(DRIVE_ROOT, zs["embedding_file"]),
    "aggregated": os.path.join(DRIVE_ROOT, agg["embedding_file"]),
    **{SLUG[s]: os.path.join(DRIVE_ROOT, ps["per_study"][s]["embedding_file"]) for s in STUDIES},
}
# L0: the zero-shot + aggregated pair already exist (colab_12, full population); the three
# per-study ones do not. Cache names bake in SUB_CAP and SEED so a changed subsample can never
# silently reload a stale embedding (the colab_13 cache-key lesson).
evals_rec = audit["geneformer_cpt_evals"]["embedding_files"]
L0_PATHS = {
    "zeroshot":   os.path.join(DRIVE_ROOT, evals_rec["zeroshot_L0"]),
    "aggregated": os.path.join(DRIVE_ROOT, evals_rec["cpt_L0"]),
    **{SLUG[s]: os.path.join(
        GF_DIR, f"glia_geneformer_cpt_per_study_{SLUG[s]}_seed0_L0_cap{SUB_CAP}s{SEED}{SUFFIX}.h5ad")
       for s in STUDIES},
}

print("\nL-1 embeddings (tier 1 -- all must exist):")
for m in MODELS:
    print(f"  {m:12s} {'OK     ' if os.path.exists(L1_PATHS[m]) else 'MISSING'} "
          f"{os.path.relpath(L1_PATHS[m], DRIVE_ROOT)}")
_missing = [m for m in MODELS if not os.path.exists(L1_PATHS[m])]
assert not _missing, f"missing L-1 embeddings for {_missing} -- colab_09/11/14 outputs must be on Drive"

print("\nL0 embeddings (tier 2):")
for m in MODELS:
    state = "OK     " if os.path.exists(L0_PATHS[m]) else ("to build" if EXTRACT_L0 else "absent ")
    print(f"  {m:12s} {state} {os.path.relpath(L0_PATHS[m], DRIVE_ROOT)}")
if EXTRACT_L0:
    for m in ("zeroshot", "aggregated"):
        assert os.path.exists(L0_PATHS[m]), (
            f"tier 2 re-uses colab_12's {m} L0 embedding; not found at {L0_PATHS[m]}")

print("\ndetector #1 drift_all (population-matched over all 142,588 cells; colab_14 / colab_11):")
for m in MODELS[1:]:
    print(f"  {m:12s} drift_all {DRIFT_ALL[m]:.5f} | {STEPS[m]:5d} steps | {N_TRAIN[m]:6d} train cells")

embedding Geneformer commit: 04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5

L-1 embeddings (tier 1 -- all must exist):
  zeroshot     OK      geneformer/glia_geneformer_zeroshot.h5ad
  aggregated   OK      geneformer/glia_geneformer_cpt_aggregated_v2_seed0.h5ad
  seaad        OK      geneformer/glia_geneformer_cpt_per_study_seaad_seed0.h5ad
  li2025       OK      geneformer/glia_geneformer_cpt_per_study_li2025_seed0.h5ad
  haney2024    OK      geneformer/glia_geneformer_cpt_per_study_haney2024_seed0.h5ad

L0 embeddings (tier 2):
  zeroshot     OK      geneformer/glia_geneformer_zeroshot_L0.h5ad
  aggregated   OK      geneformer/glia_geneformer_cpt_aggregated_v2_seed0_L0.h5ad
  seaad        absent  geneformer/glia_geneformer_cpt_per_study_seaad_seed0_L0_cap40s0.h5ad
  li2025       absent  geneformer/glia_geneformer_cpt_per_study_li2025_seed0_L0_cap40s0.h5ad
  haney2024    absent  geneformer/glia_geneformer_cpt_per_study_haney2024_seed0_L0_cap40s0.h5ad

detector #1 drift_all (population-match

> **Interpretation — all required embeddings located, drift numbers read from the recorded trail (4a).** All five L−1 embedding files (zero-shot, aggregated, and the three per-study checkpoints) were found on Drive, each carrying the same Geneformer commit (`04c2b2e8...`) — meaning every embedding compared below was produced under one shared tokenizer/rank-encoding, so a difference between rows can only mean "the model differed," not "the tokenizer also differed." The L0 (last-layer) inventory shows the two files reused from colab_12 (zero-shot, aggregated) present, and all three per-study L0 files correctly absent — expected under `EXTRACT_L0=False`, since those embeddings only get created inside the tier-2 branch, which this run does not take. The drift numbers printed here — aggregated 0.00505 over 2,000 steps / 94,963 train cells (read from the `geneformer_cpt_aggregated_v2` entry colab_11 wrote); SEA-AD 0.00543 over 1,079 steps / 51,218 cells, Li2025 0.00300 over 660 steps / 31,349 cells, Haney2024 0.00359 over 261 steps / 12,396 cells (all three read from the `geneformer_cpt_per_study` entry colab_14 wrote) — are read directly out of the recorded audit trail, not recomputed here. These are the same population-matched `drift_all` numbers that established a genuine 1.81x spread between the highest (SEA-AD) and lowest (Li2025) per-study checkpoint, now carried forward as the reference point Section 8 compares the eval deltas against. Worth being precise about units: drift here is `1 − cos(θ)`, which scales roughly with the square of the angle between two embeddings, not linearly — so "1.81x" is a ratio of that squared-ish quantity, not a 1.81x difference in the actual angle between checkpoints (which would be closer to √1.81 ≈ 1.35x). Nothing downstream in this notebook treats the ratio as if it were linear, so this is a units clarification, not a bug — but it's the kind of mistake this project has made once before with drift quantities, worth stating precisely rather than implying "1.81x more different" in angle terms.

### 4b — Define the tier-2 subsample and check its power **before** spending any GPU time

Deliberate ordering: the subsample is drawn and its held-out composition checked here, *before* 4c/4d tokenize and embed. A subsample that starves an eval class is a wasted GPU run, and the only way to find out cheaply is to look first — the same "subsample before the dominant cost" invariant colab_13 arrived at.

Two pre-committed power floors, both applied to the **held-out** set, and both fixed here before any result is seen (not tuned to one afterwards):

- `MIN_TEST_DONORS = 3` — carried unchanged from colab_12/13. Fewer than three held-out donors and the metric is one donor's idiosyncrasy.
- `MIN_TEST_CELLS = 100` — **new, and it needs a call before this notebook is trusted** (see the note below). Balanced accuracy weights every class equally regardless of size, so a thin class injects non-cancelling noise into the delta. The floor is **not** derived from an i.i.d.-cells binomial SE (`sqrt(0.25/n)`) — that would contradict this section's own donor-is-the-true-unit argument above, and it is the wrong quantity besides (the SE of an absolute recall, not of a *paired* difference between two balanced accuracies scored on identical cells, which is smaller). The real justification is coarser and stated plainly: 100 is a floor against a class dominated by a single donor (at `SUB_CAP=40`, one donor could otherwise supply most of a class's held-out cells) and against a class thin enough that a handful of mispredicted cells swings the balanced accuracy by several points. A plausibility floor, not a calibrated statistical bound — sign-off should be sought on that basis.

The second floor closes a known gap in colab_12: its `UNDERPOWERED` gate checked donor count only, not the contract's "near-absent **or** single-donor" cell criterion. It never triggered on the full population, which is why it could be deferred — but tier 2 evaluates a subsample, which is exactly the "thinner strata" case that makes it load-bearing. It is implemented here so it cannot be forgotten; the specific value of 100 is a proposal derived from the band width, and should be signed off (or replaced) rather than inherited silently.

In [7]:
MIN_TEST_DONORS = 3     # carried from colab_12/13 unchanged
MIN_TEST_CELLS  = 100   # NEW -- see 4b; a plausibility floor (single-donor-dominated / thin-class
                        # noise), NOT a calibrated SE bound -- needs sign-off

APOE_KEEP = {"carrier", "noncarrier"}                                    # e2 excluded per the locked E4 definition
BINARY    = {"microglia": ("homeostatic", "activated"),
             "astrocyte": ("resting", "reactive")}                       # intermediate excluded from the probe

# Tier-2 subsample: cap per (donor x lineage x substate). Capping at DONOR granularity is the point --
# apoe_carrier is a donor-level property, so every donor surviving keeps eval #2's class composition
# intact, and the effective sample size of a donor-held-out eval is the donor count, not the cell count.
_keep_sub = (glia.obs.groupby(["donor_id", "lineage", "substate"], observed=True)
             .apply(lambda g: g.sample(min(len(g), SUB_CAP), random_state=SEED))
             .index.get_level_values(-1))
SUB_MASK = glia.obs.index.isin(_keep_sub)
obs_sub  = glia.obs[SUB_MASK].copy()

print(f"tier-2 subsample: {int(SUB_MASK.sum())} / {glia.n_obs} cells "
      f"({100 * SUB_MASK.mean():.1f}%) at SUB_CAP={SUB_CAP}")
print("  donors kept:", obs_sub["donor_id"].nunique(), "of", glia.obs["donor_id"].nunique())
print("  split:", obs_sub["split"].value_counts().to_dict())
print("  study:", obs_sub["study_id"].value_counts().to_dict())

def _power_preview(o, label):
    """Print held-out class counts against both floors. Warn only -- the formal gate is 6a/7a."""
    print(f"\n[{label}] held-out class counts vs floors "
          f"(donors >= {MIN_TEST_DONORS}, cells >= {MIN_TEST_CELLS}):")
    thin = []
    te = o[o["split"] == "test"]
    for lin, classes in BINARY.items():
        for cls in classes:
            s = te[(te["lineage"] == lin) & (te["substate"] == cls)]
            bad = s["donor_id"].nunique() < MIN_TEST_DONORS or len(s) < MIN_TEST_CELLS
            thin += [f"eval1 {lin}/{cls}"] if bad else []
            print(f"  eval1 {lin:10s} {cls:12s}: {len(s):6d} cells | {s['donor_id'].nunique():3d} donors"
                  f"{'   <-- THIN' if bad else ''}")
    for lin in BINARY:
        for cls in sorted(APOE_KEEP):
            s = te[(te["lineage"] == lin) & (te["apoe_carrier"] == cls)]
            bad = s["donor_id"].nunique() < MIN_TEST_DONORS or len(s) < MIN_TEST_CELLS
            thin += [f"eval2 {lin}/{cls}"] if bad else []
            print(f"  eval2 {lin:10s} {cls:12s}: {len(s):6d} cells | {s['donor_id'].nunique():3d} donors"
                  f"{'   <-- THIN' if bad else ''}")
    return thin

thin_full = _power_preview(glia.obs, "tier 1 / full population")
thin_sub  = _power_preview(obs_sub,  f"tier 2 / subsample (cap {SUB_CAP})")

_sub_only_thin = sorted(set(thin_sub) - set(thin_full))
if _sub_only_thin:
    print(f"\nWARNING: the subsample makes {_sub_only_thin} underpowered even though it is fine on "
          f"the full population. Raise SUB_CAP and re-run this cell BEFORE spending GPU time on 4c/4d; "
          f"otherwise tier 2 will spend a GPU pass to produce a verdict labelled 'underpowered'.")
if thin_full:
    print(f"\nWARNING: {sorted(set(thin_full))} are underpowered on the FULL population -- not a "
          f"subsampling artifact. These will be reported as underpowered, never as a win.")
if not _sub_only_thin and not thin_full:
    print("\nno class falls below either floor, in either tier.")

tier-2 subsample: 25440 / 142588 cells (17.8%) at SUB_CAP=40
  donors kept: 145 of 145
  split: {'train': 17618, 'test': 3928, 'val': 3894}
  study: {'SEA-AD': 11753, 'Li2025': 9135, 'Haney2024': 4552}

[tier 1 / full population] held-out class counts vs floors (donors >= 3, cells >= 100):
  eval1 microglia  homeostatic :   5538 cells |  21 donors
  eval1 microglia  activated   :   1883 cells |  22 donors
  eval1 astrocyte  resting     :   6046 cells |  22 donors
  eval1 astrocyte  reactive    :   4199 cells |  22 donors
  eval2 microglia  carrier     :   4198 cells |  11 donors
  eval2 microglia  noncarrier  :   4910 cells |  10 donors
  eval2 astrocyte  carrier     :   7009 cells |  11 donors
  eval2 astrocyte  noncarrier  :   6439 cells |  10 donors

[tier 2 / subsample (cap 40)] held-out class counts vs floors (donors >= 3, cells >= 100):
  eval1 microglia  homeostatic :    786 cells |  21 donors
  eval1 microglia  activated   :    612 cells |  22 donors
  eval1 astrocyte  resting 

> **Interpretation — every eval class clears both power floors, in both tiers (4b).** The tier-2 subsample (were it to run) would keep 25,440 of 142,588 cells (17.8%) at a cap of 40 per donor×lineage×substate, and critically keeps all 145 of 145 donors — the point of capping at donor granularity rather than dropping donors outright, since these evals are donor-held-out and a donor is the real unit of independent evidence, not a cell. The power check that matters for this actual run is the "tier 1 / full population" block: every eval#1 and eval#2 class clears both floors (≥3 donors, ≥100 cells) comfortably. The thinnest class by far is microglia "activated" at 1,883 cells across 22 donors — still nearly 19x over the 100-cell floor. The `MIN_TEST_CELLS=100` floor is new in this notebook. It is a plausibility floor, not a statistically derived threshold — deliberately so: because these evals are donor-held-out, the true independent unit is the donor, not the cell, so an i.i.d.-cells binomial standard-error calculation would contradict the notebook's own reasoning about what the effective sample size is. The value guards against a cruder failure mode instead: a class that is near-absent or effectively carried by a single donor's cells, which the donor-count floor (`MIN_TEST_DONORS=3`) alone would not catch. 100 was chosen as a round, plausible order of magnitude for that purpose, not derived from a formula, and is recorded in `docs/EVALUATION_CONTRACT.md` as exactly that kind of floor. Because nothing tripped either floor here, none of the verdicts below are forced to "underpowered"; every eval class had a genuine chance to show a real effect if one existed.

### 4c — Tokenize the subsample (tier 2 only; identical per-cell encoding to every prior run)

Only the subsample is tokenized, which is what makes tier 2 cheap. This is safe because Geneformer's encoding is **per cell**: a cell's rank-value token sequence is a function of its own counts alone, not of which other cells are in the dataset. So tokenizing a subset yields byte-identical tokens to tokenizing the full set and slicing — the same property colab_12's SMOKE path already relied on.

The raw-counts guard lives here rather than in 2a because this is the only place it matters: Geneformer rank-encodes **raw** counts, so a normalized `.X` would silently produce a different encoding than every embedding it is about to be compared against. `sum_ensembl_ids` gets the same `RangeIndex` monkeypatch colab_11 needed, and APOE must be in the vocabulary or eval #2 is impossible by construction (a pre-registered hard fail).

In [8]:
if not EXTRACT_L0:
    print("EXTRACT_L0=False -- skipping tokenization (tier 1 needs no new embeddings).")
else:
    import pickle
    from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE, TranscriptomeTokenizer
    import geneformer.tokenizer as _gftok

    glia_sub = glia[SUB_MASK].copy()

    # Geneformer rank-encodes RAW counts -- a normalized .X would encode differently from every
    # embedding this will be compared against.
    _idx = np.random.default_rng(0).choice(glia_sub.n_obs, size=min(2000, glia_sub.n_obs), replace=False)
    _Xs  = glia_sub.X[_idx]
    _dat = _Xs.data if sp.issparse(_Xs) else np.asarray(_Xs).ravel()
    _frac_int = float(np.mean(np.mod(_dat, 1) == 0)) if _dat.size else 1.0
    assert _frac_int >= 0.99, f".X is not raw counts (integer fraction {_frac_int:.3f})"

    with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
        symbol_to_ensembl = pickle.load(f)
    with open(TOKEN_DICTIONARY_FILE, "rb") as f:
        token_dictionary = pickle.load(f)

    glia_sub.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in glia_sub.var_names]
    in_vocab = glia_sub.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)
    apoe_e = symbol_to_ensembl.get("APOE")
    assert apoe_e is not None and apoe_e in token_dictionary, (
        "APOE not tokenizable -- eval #2 is impossible for Geneformer (pre-registered hard fail)")
    print(f"genes in Geneformer vocab: {int(in_vocab.sum())} / {glia_sub.n_vars}")

    glia_sub.obs["n_counts"] = np.asarray(glia_sub.X.sum(axis=1)).ravel()
    assert (glia_sub.obs["n_counts"] > 0).all(), "cells with zero counts present"

    TOK_IN_DIR  = f"/content/gf_token_in_c15{SUFFIX}"
    TOK_OUT_DIR = f"/content/gf_token_out_c15{SUFFIX}"
    os.makedirs(TOK_IN_DIR, exist_ok=True); os.makedirs(TOK_OUT_DIR, exist_ok=True)
    glia_sub.write_h5ad(os.path.join(TOK_IN_DIR, "glia_sub.h5ad"))

    LABEL_COLS   = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
    CUSTOM_ATTRS = {c: c for c in LABEL_COLS}

    # colab_11's monkeypatch: tokenize_anndata positional-indexes var by an integer array against a
    # non-integer index; reset it so upstream's assumed behaviour holds.
    _orig_sum = _gftok.sum_ensembl_ids
    def _sum_rangeindex_patch(*a, **k):
        r = _orig_sum(*a, **k); r.var.index = pd.RangeIndex(r.n_vars); return r
    _gftok.sum_ensembl_ids = _sum_rangeindex_patch

    tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
    tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, f"glia_c15_sub{SUFFIX}", file_format="h5ad")
    TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, f"glia_c15_sub{SUFFIX}.dataset")

    # Geneformer's tokenizer can silently drop cells below its gene-count floor; a shorter dataset
    # than expected would misalign every embedding produced from it (colab_14 M2).
    from datasets import load_from_disk
    _ds = load_from_disk(TOKENIZED_DATASET)
    assert len(_ds) == glia_sub.n_obs, \
        f"tokenizer returned {len(_ds)} cells for {glia_sub.n_obs} input cells -- cells were dropped"
    print("tokenized ->", TOKENIZED_DATASET, f"({len(_ds)} cells)")

EXTRACT_L0=False -- skipping tokenization (tier 1 needs no new embeddings).


> **Interpretation — tokenization correctly skipped (4c).** This cell only does work when `EXTRACT_L0=True`, since tier 1 needs no new embeddings — every matrix it requires already exists on Drive from earlier notebooks. The one-line print confirms the branch was taken as designed; nothing was tokenized, and no Geneformer forward pass has happened at any point in this run so far.

### 4d — Extract `emb_layer=0` for each per-study checkpoint (tier 2 only)

One extraction pass per per-study adapter: load a **fresh** frozen base, merge that adapter in, save the merged weights, run `EmbExtractor` at `emb_layer=0`, realign to `cell_index`, write. Fresh base every time so no adapter can leak into the next — the same hygiene colab_14's `run_study()` used.

Two inherited hazards, both handled rather than assumed away:

- **Attention backend pinned to eager.** `transformers` may default BERT to a fused SDPA kernel; `geneformer.perturber_utils.load_model` resolves `BertForMaskedLM` from module globals at call time, including inside `EmbExtractor`, so the patch has to be applied there. colab_14's fbs-safety comment *assumed* eager while nothing in that notebook actually pinned it — pinning it here makes the geometry bound below true rather than hopeful.
- **Batch size derived, not hardcoded.** Eager attention materializes a `(B, heads, L, L)` score tensor; glia nuclei tokenize to Geneformer V2's full 4096-token ceiling, where `64 x 12 x 4096^2` overflows signed int32 and faults on the first kernel of the forward. `FWD_BATCH` halves from 64 until the product fits. Batch size only changes cost, not results — embeddings are batch-invariant in eval mode.

- **Reused vs. newly-built L0 files are not produced under identical config.** colab_12's zero-shot/aggregated L0 embeddings (reused here, not rebuilt) were extracted at `forward_batch_size=64` with no eager pin — safe only because their max tokenized length happened to stay under the int32 threshold this section derives explicitly. The three per-study L0 files built here use eager attention and a smaller, adaptively-derived `FWD_BATCH`. Embeddings are batch-invariant in eval mode and attention-backend-invariant in exact arithmetic, so this is expected to be floating-point-noise-level, not a real confound — but it is a genuine config difference between the reused pair and the new pair that nothing in this notebook checks directly.

Each write is exists-guarded, so an interrupted run resumes instead of repeating a completed pass.

In [9]:
if not EXTRACT_L0:
    print("EXTRACT_L0=False -- skipping the L0 extraction passes (tier 2 disabled).")
else:
    import torch
    from transformers import BertForMaskedLM, BertConfig
    from peft import PeftModel
    from geneformer import EmbExtractor
    import geneformer.perturber_utils as _pu

    # Pin eager attention. EmbExtractor reaches BertForMaskedLM through perturber_utils' module
    # globals at call time, so this is the interception point that actually takes effect.
    class _EagerBertForMaskedLM:
        @staticmethod
        def from_pretrained(*a, **k):
            k.setdefault("attn_implementation", "eager")
            return BertForMaskedLM.from_pretrained(*a, **k)
    _pu.BertForMaskedLM = _EagerBertForMaskedLM

    MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
    assert os.path.exists(MODEL_DIR), f"base model missing: {MODEL_DIR}"

    # Verify the monkeypatch actually resolves attention to eager rather than silently falling back
    # to transformers' unpinned default -- the cheap check docs/ASSUMPTIONS.md's open "why did
    # colab_14 complete cleanly under a backend colab_13 declared always-faulting" item names
    # directly. Cost: one extra base-model load through the patched path.
    _attn_probe = _pu.BertForMaskedLM.from_pretrained(MODEL_DIR)
    _resolved = getattr(_attn_probe.config, "_attn_implementation", None)
    assert _resolved == "eager", (
        f"attn_implementation resolved to {_resolved!r} via the patched loader, not 'eager' -- "
        f"the int32-safety derivation below assumes eager materialises the full (B,heads,L,L) "
        f"score tensor and is not valid under a fused SDPA backend")
    print(f"attn_implementation via the patched loader: {_resolved!r} (verified)")
    del _attn_probe

    _cfg = BertConfig.from_pretrained(MODEL_DIR)

    # --- derive an int32-safe forward batch against THIS run's own token lengths ---
    MAX_LEN  = int(max(_ds["length"]))
    N_HEADS  = _cfg.num_attention_heads
    INT32MAX = 2**31 - 1
    assert MAX_LEN <= _cfg.max_position_embeddings, (
        f"tokenized length {MAX_LEN} exceeds max_position_embeddings {_cfg.max_position_embeddings}")
    FWD_BATCH = 64
    while FWD_BATCH > 1 and FWD_BATCH * N_HEADS * MAX_LEN**2 >= INT32MAX:
        FWD_BATCH //= 2
    assert FWD_BATCH * N_HEADS * MAX_LEN**2 < INT32MAX, \
        f"even fbs=1 overflows int32 at max_len={MAX_LEN}, heads={N_HEADS}"
    n_trunc = int(sum(1 for L in _ds["length"] if L >= _cfg.max_position_embeddings))
    print(f"fbs={FWD_BATCH} | max_len={MAX_LEN} (ceiling {_cfg.max_position_embeddings}), heads={N_HEADS}, "
          f"attention tensor {FWD_BATCH * N_HEADS * MAX_LEN**2:,} < int32 {INT32MAX:,}")
    print(f"cells tokenized at the {_cfg.max_position_embeddings}-token ceiling (truncated): "
          f"{n_trunc} / {len(_ds)}")

    def _extract_L0(model_dir, tag, out_h5ad):
        ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                          max_ncells=None, emb_layer=0, emb_label=LABEL_COLS,
                          forward_batch_size=FWD_BATCH, nproc=4)
        work = f"/content/gf_emb_c15_{tag}{SUFFIX}"; os.makedirs(work, exist_ok=True)
        df = ee.extract_embs(model_dir, TOKENIZED_DATASET, work, f"glia_c15_{tag}")
        emb_cols = [c for c in df.columns if c not in LABEL_COLS]
        df = df.set_index("cell_index").reindex(glia_sub.obs["cell_index"].values)
        assert df[emb_cols].notna().all().all(), f"{tag}: rows missing after cell_index realignment"
        a = ad.AnnData(X=df[emb_cols].to_numpy(dtype=np.float32),
                       obs=glia_sub.obs[LABEL_COLS].reset_index(drop=True))
        a.write_h5ad(out_h5ad)
        print(f"  {tag} L0 -> {os.path.basename(out_h5ad)}  {a.shape}")

    for study in STUDIES:
        slug, out = SLUG[study], L0_PATHS[SLUG[study]]
        if os.path.exists(out):
            print(f"[{study}] L0 exists, skipping: {os.path.basename(out)}")
            continue
        adapter = ADAPTERS[slug]
        assert os.path.exists(adapter), f"[{study}] adapter missing on Drive: {adapter}"
        print(f"[{study}] merging {os.path.relpath(adapter, DRIVE_ROOT)} into a fresh base")
        base   = BertForMaskedLM.from_pretrained(MODEL_DIR, attn_implementation="eager")
        merged = PeftModel.from_pretrained(base, adapter).merge_and_unload()
        merged_dir = f"/content/gf_merged_c15_{slug}{SUFFIX}"; os.makedirs(merged_dir, exist_ok=True)
        merged.save_pretrained(merged_dir)
        base.config.to_json_file(os.path.join(merged_dir, "config.json"))
        del base, merged; gc.collect(); torch.cuda.empty_cache()
        _extract_L0(merged_dir, f"cpt_per_study_{slug}", out)
        gc.collect(); torch.cuda.empty_cache()

    for m in MODELS:
        assert os.path.exists(L0_PATHS[m]), f"L0 embedding still missing for {m}: {L0_PATHS[m]}"
    print("\nall five L0 embeddings present.")

EXTRACT_L0=False -- skipping the L0 extraction passes (tier 2 disabled).


> **Interpretation — L0 extraction correctly skipped, zero GPU passes this run (4d).** Like 4c, this cell's actual work (three fresh-base-plus-adapter merge-and-embed passes, one per per-study checkpoint) only runs under `EXTRACT_L0=True`. The skip print confirms it took the tier-1 branch. Combined with 4c, this closes out the tier-2 half of the notebook entirely for this run: no tokenization and no embedding extraction happened anywhere, so the GPU attached to this session (flagged in 1b) went completely unused start to finish — every number produced from here on comes from loading and analyzing files that already existed before this run started.

## 5 — Assemble the embedding matrices and the shared eval machinery

### 5a — Load and align every matrix to `cell_index`; define bands and metrics once

Matrices are keyed `(tier, model, layer)`. Tier 1 holds five L-1 matrices over the full population; tier 2, when enabled, holds five more at each of L-1 and L0 over the subsample — the L-1 pair reuses the same files as tier 1, sliced to the subsample, so the −1→L0 contrast is on identical cells. Alignment is by `cell_index` reindex with a completeness assert, never by row order.

The contract bands are defined here once, unchanged from colab_12: eval #1 and eval #2's k-NN share the accuracy-gain shape (noise / meaningful >= 5 pp / decisive >= 10 pp) differing only in the regression cutoff (−2 pp vs −3 pp), and silhouette has its own band. Silhouette stays **corroborating-only** at these donor counts — k-NN is the load-bearing metric for eval #2.

The cell also drops the raw count matrices once `OBS` is extracted. Five full-population embedding matrices are ca. 2.2 GB and each alignment makes a transient copy, so holding the sparse counts at the same time can exhaust a standard Colab runtime. The cost is that this cell is not idempotent — re-running it alone will fail, and 2a/3a have to be re-run first, which the normal top-to-bottom order does anyway.

In [10]:
def _load_aligned(path, index_values, tag):
    a = sc.read_h5ad(path)
    X = a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X)
    df = pd.DataFrame(X, index=a.obs["cell_index"].values).reindex(index_values)
    assert df.notna().all().all(), f"{tag}: rows missing after cell_index alignment ({path})"
    del a, X; gc.collect()
    return df.to_numpy(dtype=np.float32)

TIERS  = ["full"] + (["sub"] if EXTRACT_L0 else [])
LAYERS = {"full": ["L-1"], "sub": ["L-1", "L0"]}
OBS    = {"full": glia.obs.reset_index(drop=True), "sub": obs_sub.reset_index(drop=True)}
IDX    = {t: OBS[t]["cell_index"].values for t in ("full", "sub")}

# Free the raw count matrices BEFORE loading the embeddings. Five full-population matrices are
# ca. 2.2 GB and each _load_aligned makes a transient copy while reindexing, so holding the sparse
# counts as well can exhaust a standard Colab runtime. Everything downstream needs only `OBS` and
# `EMB`. NOTE: this makes the cell non-idempotent -- re-running it requires re-running 2a/3a first
# (the normal top-to-bottom order does this anyway).
for _n in ("glia", "glia_sub"):
    if globals().get(_n) is not None:
        del globals()[_n]
gc.collect()

EMB = {}
for m in MODELS:
    EMB[("full", m, "L-1")] = _load_aligned(L1_PATHS[m], IDX["full"], f"full/{m}/L-1")
if EXTRACT_L0:
    for m in MODELS:
        EMB[("sub", m, "L-1")] = _load_aligned(L1_PATHS[m], IDX["sub"], f"sub/{m}/L-1")
        EMB[("sub", m, "L0")]  = _load_aligned(L0_PATHS[m], IDX["sub"], f"sub/{m}/L0")

for k in sorted(EMB, key=str):
    print(k, EMB[k].shape)
for t in TIERS:
    shapes = {EMB[(t, m, l)].shape for m in MODELS for l in LAYERS[t]}
    assert len(shapes) == 1, f"tier {t}: embedding matrices differ in shape ({shapes})"

def _per_cell_cosine(A, B):
    # matches colab_14's detector #1 formula exactly, so the checks below are computed the same
    # way as every drift number already sitting in audit_report.json
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12
    return num / den

# Verify drift_all wasn't computed from a different embedding file than the one just loaded here --
# audit_report.json points to a path, but nothing upstream confirms the file living at that path
# TODAY is still the one that produced the recorded number (a stale overwrite would go undetected
# by the notna/shape checks above, which pass on any complete, correctly-shaped, WRONG matrix).
_zs_full = EMB[("full", "zeroshot", "L-1")]
_test_mask_full = (OBS["full"]["split"] == "test").to_numpy()
DRIFT_GLOBAL_TEST = {}
for m in MODELS[1:]:
    _cos = _per_cell_cosine(_zs_full, EMB[("full", m, "L-1")])
    _drift_all_check = 1.0 - float(np.median(_cos))
    _d = abs(_drift_all_check - DRIFT_ALL[m])
    assert _d <= 0.0005, (
        f"{m}: drift_all recomputed from the loaded embedding ({_drift_all_check:.5f}) disagrees "
        f"with audit_report.json's recorded value ({DRIFT_ALL[m]:.5f}) by {_d:.5f} -- the file at "
        f"L1_PATHS[{m!r}] may not be the one that produced the recorded number")
    # drift on the SAME population the evals below actually score (global held-out test, pooled
    # across studies) -- drift_all (all 142,588 cells) is a DIFFERENT population, and 8a's
    # ordering-as-prediction reading needs the eval-matched number (drift_test, the other number
    # colab_14 recorded, is worse still: a different population per study).
    DRIFT_GLOBAL_TEST[m] = 1.0 - float(np.median(_cos[_test_mask_full]))
print("drift_all recomputed from the loaded embeddings matches audit_report.json for all checkpoints.")
print("drift on the global held-out test split (the population evals #1/#2 actually score):")
for m in MODELS[1:]:
    print(f"  {m:12s} drift_global_test {DRIFT_GLOBAL_TEST[m]:.5f}  (drift_all {DRIFT_ALL[m]:.5f})")

# --- contract bands (colab_12, unchanged) ---
def band_probe(d_pp):        # eval #1
    if d_pp < -2:  return "regression"
    if d_pp >= 10: return "decisive"
    if d_pp >= 5:  return "meaningful"
    return "noise"

def band_knn(d_pp):          # eval #2, load-bearing
    if d_pp < -3:  return "regression"
    if d_pp >= 10: return "decisive"
    if d_pp >= 5:  return "meaningful"
    return "noise"

def band_sil(d):             # eval #2, corroborating only
    if d < 0:      return "regression"
    if d >= 0.10:  return "decisive"
    if d >= 0.05:  return "meaningful"
    return "noise"

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, silhouette_score

def probe_bacc(X, tr, te, y):
    s = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(s.transform(X[tr]), y[tr])
    return balanced_accuracy_score(y[te], clf.predict(s.transform(X[te])))

def apoe_metrics(X, tr, te, y):
    s = StandardScaler().fit(X[tr])
    Xtr, Xte = s.transform(X[tr]), s.transform(X[te])
    knn = KNeighborsClassifier(n_neighbors=15).fit(Xtr, y[tr])
    bacc = balanced_accuracy_score(y[te], knn.predict(Xte))
    sil  = silhouette_score(Xte, y[te]) if len(np.unique(y[te])) == 2 else float("nan")
    return bacc, sil

print(f"\ntiers active: {TIERS} | models: {MODELS}")

('full', 'aggregated', 'L-1') (142588, 768)
('full', 'haney2024', 'L-1') (142588, 768)
('full', 'li2025', 'L-1') (142588, 768)
('full', 'seaad', 'L-1') (142588, 768)
('full', 'zeroshot', 'L-1') (142588, 768)
drift_all recomputed from the loaded embeddings matches audit_report.json for all checkpoints.
drift on the global held-out test split (the population evals #1/#2 actually score):
  aggregated   drift_global_test 0.00506  (drift_all 0.00505)
  seaad        drift_global_test 0.00556  (drift_all 0.00543)
  li2025       drift_global_test 0.00300  (drift_all 0.00300)
  haney2024    drift_global_test 0.00363  (drift_all 0.00359)

tiers active: ['full'] | models: ['zeroshot', 'aggregated', 'seaad', 'li2025', 'haney2024']


> **Interpretation — five matrices loaded and aligned, drift recompute is genuinely non-circular (5a).** All five L−1 embedding matrices loaded at the identical shape `(142588, 768)` — one 768-dimensional vector per cell, aligned by reindexing on `cell_index` rather than trusted row order, with a completeness assert behind that reindex. The line "drift_all recomputed from the loaded embeddings matches audit_report.json for all checkpoints" is the notebook re-deriving each checkpoint's cosine drift to zero-shot directly from these raw vectors and only then comparing to the stored value — a genuine independent check, not a reload-and-compare-to-itself circularity (the kind of vacuous check a review caught in an earlier notebook in this project). The new `drift_global_test` numbers (drift measured only on the population evals #1/#2 actually score, rather than all 142,588 cells) track `drift_all` closely for every checkpoint (e.g. aggregated 0.00506 vs. 0.00505, SEA-AD 0.00556 vs. 0.00543) — and, importantly, preserve the same high-to-low ordering (SEA-AD > Haney2024 > Li2025) in both populations. That ordering stability is what makes the drift-vs-eval comparison in 8a meaningful to run at all: if the ranking had flipped between the full population and the eval-scored population, comparing it against eval deltas measured only on the latter would not be a fair comparison.

## 6 — Eval #1: substate linear probe

### 6a — Held-out substate composition audit (a thin substate is a null, not a win)

The contract's mandatory reporting: per lineage and per substate, how many held-out donors and how many cells carry it, and which studies they come from. A class failing either floor forces the **verdict label itself** to `underpowered` in 6b — in colab_12 the flag was printed but not inherited by the verdict table, which is exactly the "reported as underpowered, never as a win" failure the contract names, so the gate returns a value that 6b consumes rather than a message a reader has to notice.

Reminder on what the class labels mean here: microglia `activated` and astrocyte `reactive` are ca. 69% and ca. 79% donor-study confounded upstream, so a probe gain on this axis is not automatically a biological read. Nothing on this run hinges on that — but it is the reason a *win* here would need the composition table read alongside it.

In [11]:
def substate_power(o, label):
    """Print held-out substate composition; return {lineage: underpowered_bool} for 6b to consume."""
    flags = {}
    print(f"=== held-out substate composition [{label}] ===")
    for lin, (neg, pos) in BINARY.items():
        m   = (o["split"] == "test") & (o["lineage"] == lin)
        sub = o.loc[m, ["substate", "donor_id", "study_id"]]
        print(f"\n[{lin}] binary classes = {neg} vs {pos}  (intermediate excluded from the probe)")
        under = False
        for s in (neg, pos, "intermediate"):
            ss = sub[sub["substate"] == s]
            nd = ss["donor_id"].nunique()
            if s == "intermediate":
                tag = ""
            else:
                thin_d, thin_c = nd < MIN_TEST_DONORS, len(ss) < MIN_TEST_CELLS
                under = under or thin_d or thin_c
                tag = ("  [UNDERPOWERED: donors]" if thin_d else "") + \
                      ("  [UNDERPOWERED: cells]" if thin_c else "")
            print(f"  {s:12s}: {len(ss):6d} cells | {nd:3d} donors | "
                  f"{ss['study_id'].value_counts().to_dict()}{tag}")
        flags[lin] = under
    return flags

POWER_1 = {t: substate_power(OBS[t], t) for t in TIERS}
print("\neval #1 underpowered flags:", POWER_1)

=== held-out substate composition [full] ===

[microglia] binary classes = homeostatic vs activated  (intermediate excluded from the probe)
  homeostatic :   5538 cells |  21 donors | {'SEA-AD': 2843, 'Li2025': 2159, 'Haney2024': 536}
  activated   :   1883 cells |  22 donors | {'SEA-AD': 1389, 'Li2025': 250, 'Haney2024': 244}
  intermediate:   2508 cells |  21 donors | {'SEA-AD': 1431, 'Li2025': 802, 'Haney2024': 275}

[astrocyte] binary classes = resting vs reactive  (intermediate excluded from the probe)
  resting     :   6046 cells |  22 donors | {'Li2025': 2930, 'SEA-AD': 2578, 'Haney2024': 538}
  reactive    :   4199 cells |  22 donors | {'SEA-AD': 3023, 'Haney2024': 802, 'Li2025': 374}
  intermediate:   3627 cells |  19 donors | {'SEA-AD': 2241, 'Haney2024': 765, 'Li2025': 621}

eval #1 underpowered flags: {'full': {'microglia': False, 'astrocyte': False}}


> **Interpretation — held-out substate composition, both lineages well-powered (6a).** For microglia (homeostatic vs. activated) and astrocyte (resting vs. reactive), every one of the four binary classes clears the power floor by a wide margin — 21-22 distinct donors per class — and the underpowered flags for both lineages come back `False`. The `intermediate` bucket (2,508 microglia cells / 21 donors; 3,627 astrocyte cells / 19 donors) is printed for transparency but is excluded from the binary probe run in 6b. Worth reading alongside this table: microglia "activated" here draws 1,389 of its 1,883 cells from SEA-AD (about 74%), and astrocyte "reactive" draws 3,023 of its 4,199 cells from SEA-AD (about 72%) — in the same direction as, though not identical to, the roughly 69%/79% donor-study confound documented in the full population when these substate labels were first assigned. The two numbers aren't expected to match exactly (74%/72% here are SEA-AD's share of just this held-out test slice for these two classes, a different quantity from the full-population confound statistic), but both point the same way: these two substate buckets are dominated by one study. That confound doesn't invalidate the donor-held-out probe design, but it means a real probe "win" on this axis would need to be checked against study composition before being read as biology rather than batch structure.

### 6b — Probe every checkpoint at every active extraction point, Δ vs zero-shot

One logistic-regression probe per (tier, model, layer): fit on train-split cells, scored by balanced accuracy on the 22 disjoint held-out donors, so a prediction cannot be a memorized donor identity. All five variants — zero-shot, aggregated, and the three per-study checkpoints — are fit on the *same* cells with the *same* hyperparameters, so the only thing varying across a row is which model produced the embedding. Each CPT variant's verdict is its **balanced-accuracy gain over zero-shot** against the fixed bands.

The last block is a **reproducibility cross-check against colab_12**, and it is deliberately non-circular: colab_12 ran this identical probe on these identical embedding files and recorded the result in `audit_report.json`. Nothing here reloads those numbers into the computation — they are recomputed from the embeddings and only then compared, so a match is real evidence that the substrate rebuild, split, alignment and probe are unchanged, and a mismatch means something in that chain moved. (A reloaded number matching itself would prove nothing — the circularity that made colab_14's "reproduces the stored numbers" claim vacuous.)

In [12]:
EVAL1 = {}
for tier in TIERS:
    o = OBS[tier]
    is_train = (o["split"] == "train").to_numpy()
    is_test  = (o["split"] == "test").to_numpy()
    substate = o["substate"].to_numpy()
    lineage  = o["lineage"].to_numpy()
    for lin, (neg, pos) in BINARY.items():
        in_lin = lineage == lin
        is_bin = np.isin(substate, [neg, pos])
        tr, te = is_train & in_lin & is_bin, is_test & in_lin & is_bin
        y = (substate == pos).astype(int)          # 1 = activated / reactive
        print(f"\n[{tier}/{lin}] train {int(tr.sum())} / test {int(te.sum())} cells")
        for layer in LAYERS[tier]:
            b = {m: probe_bacc(EMB[(tier, m, layer)], tr, te, y) for m in MODELS}
            print(f"  {layer}: zero-shot bacc {b['zeroshot']:.4f}")
            for m in MODELS[1:]:
                delta   = (b[m] - b["zeroshot"]) * 100
                verdict = "underpowered" if POWER_1[tier][lin] else band_probe(delta)
                EVAL1[(tier, m, lin, layer)] = {
                    "bacc_zeroshot": round(float(b["zeroshot"]), 4),
                    "bacc_cpt": round(float(b[m]), 4),
                    "delta_pp": round(float(delta), 2), "verdict": verdict}
                print(f"    {m:12s} {b[m]:.4f}  Δ{delta:+6.2f} pp  [{verdict}]")

# --- non-circular reproducibility check against colab_12's recorded aggregated result ---
if SMOKE:
    print("\n[SMOKE] skipping the colab_12 cross-check (subsampled cells, numbers not comparable).")
else:
    ref = audit["geneformer_cpt_evals"]["eval1_substate_probe"]
    print("\n=== reproducibility vs colab_12 (aggregated, L-1, full population) ===")
    REPRO_HARD, REPRO_WARN = 0.01, 0.005   # tightened from 0.02 -- the deltas being measured are
                                            # ~0.1-0.25pp, so 0.02 could hide a systematic shift
                                            # several times larger than the effect it would confound
    for lin in BINARY:
        k = f"{lin}|L-1"
        assert k in ref, f"colab_12 recorded no eval1 entry for {k}"
        for tag, key in (("zero-shot", "bacc_zeroshot"), ("aggregated", "bacc_cpt")):
            got, exp = EVAL1[("full", "aggregated", lin, "L-1")][key], ref[k][key]
            d = abs(got - exp)
            print(f"  {lin:10s} {tag:10s}: colab_15 {got:.4f} vs colab_12 {exp:.4f}  (|Δ| {d:.4f})")
            assert d <= REPRO_HARD, (
                f"{lin} {tag}: balanced accuracy {got:.4f} differs from colab_12's recorded "
                f"{exp:.4f} by {d:.4f} -- the substrate, split, alignment or probe has changed; "
                f"do NOT trust the per-study deltas above until this is explained")
            if d > REPRO_WARN:
                print(f"    WARNING: {d:.4f} is larger than a version-drift-sized difference -- "
                      f"worth explaining before this run is closed.")


[full/microglia] train 26548 / test 7421 cells
  L-1: zero-shot bacc 0.9069
    aggregated   0.9057  Δ -0.12 pp  [noise]
    seaad        0.9058  Δ -0.10 pp  [noise]
    li2025       0.9056  Δ -0.13 pp  [noise]
    haney2024    0.9079  Δ +0.10 pp  [noise]

[full/astrocyte] train 53839 / test 10245 cells
  L-1: zero-shot bacc 0.7784
    aggregated   0.7767  Δ -0.17 pp  [noise]
    seaad        0.7795  Δ +0.11 pp  [noise]
    li2025       0.7769  Δ -0.15 pp  [noise]
    haney2024    0.7784  Δ +0.01 pp  [noise]

=== reproducibility vs colab_12 (aggregated, L-1, full population) ===
  microglia  zero-shot : colab_15 0.9069 vs colab_12 0.9069  (|Δ| 0.0000)
  microglia  aggregated: colab_15 0.9057 vs colab_12 0.9057  (|Δ| 0.0000)
  astrocyte  zero-shot : colab_15 0.7784 vs colab_12 0.7784  (|Δ| 0.0000)
  astrocyte  aggregated: colab_15 0.7767 vs colab_12 0.7767  (|Δ| 0.0000)


> **Interpretation — substate probe null for every CPT checkpoint, reproducibility confirmed exactly (6b).** A logistic-regression probe was fit separately on each of the five embedding matrices (zero-shot plus four CPT variants), trained only on train-split cells and scored by balanced accuracy on the fully disjoint held-out donors. Zero-shot baselines are already high — 0.9069 microglia, 0.7784 astrocyte — meaning Geneformer's raw embedding already linearly separates these substates well before any fine-tuning. Every CPT delta, for all four checkpoints in both lineages, is tiny (−0.17pp to +0.11pp) and lands in the pre-registered "noise" band. This extends colab_12's aggregated-only null finding to all three per-study checkpoints: despite being genuinely, differently drifted from zero-shot (per 4a/5a's drift numbers), none of the four CPT variants produces a probe-detectable substate signal. The reproducibility block at the bottom is the strongest correctness signal in this cell: the zero-shot and aggregated balanced accuracies recomputed here from the raw embeddings match colab_12's independently-stored values to four decimal places exactly (`|Δ| 0.0000`, both lineages) — confirming the substrate rebuild, split, alignment, and probe are all doing exactly what they did in that earlier notebook, not merely printing numbers that happen to look right.

## 7 — Eval #2: APOE-carrier recovery (Stanton core)

### 7a — APOE composition and confound audit

The load-bearing biological axis: does CPT make E4-carrier status more recoverable *within* each lineage? A "recovery" that is really study or region leakage is a null, so the confound audit comes first.

One change from colab_12: the breakdown is grouped by **study x region x apoe_carrier**, not study x region. colab_12's write-up claimed its table showed whether a study was carrier-skewed while the groupby could not — carrier status was not in it. Nothing hinged on it there because every eval #2 verdict was null, but a future run finding a real effect would need this to be a complete check, so the missing dimension is added here rather than left for the next notebook to inherit.

Standing caveat: Haney2024's `region` is `"unknown"` for all cells, so it contributes a single region bucket.

In [13]:
def apoe_power(o, label):
    """Print held-out APOE composition + confound table; return {lineage: underpowered_bool}."""
    flags = {}
    print(f"=== held-out APOE composition [{label}] ===")
    for lin in ("microglia", "astrocyte"):
        m   = (o["split"] == "test") & (o["lineage"] == lin) & o["apoe_carrier"].isin(APOE_KEEP)
        sub = o.loc[m, ["apoe_carrier", "study_id", "region", "donor_id"]]
        print(f"\n[{lin}] held-out carrier/noncarrier cells: {len(sub)}  (e2 excluded)")
        under = False
        for cls in ("carrier", "noncarrier"):
            cc = sub[sub["apoe_carrier"] == cls]
            nd = cc["donor_id"].nunique()
            thin_d, thin_c = nd < MIN_TEST_DONORS, len(cc) < MIN_TEST_CELLS
            under = under or thin_d or thin_c
            tag = ("  [UNDERPOWERED: donors]" if thin_d else "") + \
                  ("  [UNDERPOWERED: cells]" if thin_c else "")
            print(f"  {cls:11s}: {len(cc):6d} cells | {nd:3d} donors{tag}")
        # study x region x apoe_carrier: the carrier dimension colab_12's audit was missing, so the
        # table can actually show whether a study or region is carrier-skewed.
        print("  study x region x apoe_carrier:")
        print(sub.groupby(["study_id", "region", "apoe_carrier"], observed=True).size()
                 .unstack("apoe_carrier", fill_value=0).to_string())
        flags[lin] = under
    return flags

POWER_2 = {t: apoe_power(OBS[t], t) for t in TIERS}
print("\neval #2 underpowered flags:", POWER_2)

=== held-out APOE composition [full] ===

[microglia] held-out carrier/noncarrier cells: 9108  (e2 excluded)
  carrier    :   4198 cells |  11 donors
  noncarrier :   4910 cells |  10 donors
  study x region x apoe_carrier:
apoe_carrier               carrier  noncarrier
study_id  region                              
Li2025    temporal cortex     2439         772
SEA-AD    MTG                  979        3863
Haney2024 unknown              780         275

[astrocyte] held-out carrier/noncarrier cells: 13448  (e2 excluded)
  carrier    :   7009 cells |  11 donors
  noncarrier :   6439 cells |  10 donors
  study x region x apoe_carrier:
apoe_carrier               carrier  noncarrier
study_id  region                              
Li2025    temporal cortex     3219         706
SEA-AD    MTG                 2161        5257
Haney2024 unknown             1629         476

eval #2 underpowered flags: {'full': {'microglia': False, 'astrocyte': False}}


> **Interpretation — APOE composition well-powered, and a real confound now visible for the first time (7a).** Held-out carrier/noncarrier counts (e2 excluded, per the locked APOE definition) clear the power floor comfortably in both lineages — 10-11 distinct donors per class — and both underpowered flags come back `False`. The study × region × apoe_carrier breakdown here is a genuine fix over colab_12's version of this same audit, which grouped by study × region only and therefore could not actually show whether a study was carrier-skewed. Now that carrier status is in the groupby, a real skew is visible: Li2025 is heavily carrier-leaning (microglia: 2,439 carrier vs. 772 noncarrier, about 76% carrier), SEA-AD is heavily noncarrier-leaning (microglia: 979 vs. 3,863, about 80% noncarrier), and Haney2024 leans carrier (microglia: 780 vs. 275, about 74% carrier) — the same pattern holds in astrocyte. This means study identity is a real, usable predictor of APOE-carrier status in this held-out set. It doesn't invalidate the donor-held-out design (that protects against donor-identity leakage, a different failure mode), but it means any k-NN or silhouette "win" recorded in 7b would need to be checked against this table before being trusted as genuine APOE biology rather than a study/batch-effect shortcut.

### 7b — k-NN + silhouette within astro / within micro, every checkpoint and extraction point

Two metrics reported together per the contract. **k-NN** (k=15) fits on train-split cells and predicts held-out carrier status by balanced accuracy — donor-disjoint, and the **load-bearing** metric. **Silhouette** of the binary E4 label on the held-out cells is **corroborating-only** at these donor counts: it is scored against its band, but a silhouette move alone is not a verdict — it either agrees with the k-NN read or it is noise.

Context for reading the output rather than the bands alone: colab_12 found the aggregated checkpoint's k-NN balanced accuracies sat at **0.40–0.46**, at or below the 0.50 chance line, before and after CPT. That is a floor null, not a flat one — APOE-carrier status may simply not be k-NN-recoverable from these embeddings at all. If the per-study baselines land in the same place, a near-zero delta says almost nothing about the regime; the number to look at first is the absolute balanced accuracy, not the delta.

In [14]:
EVAL2 = {}
for tier in TIERS:
    o = OBS[tier]
    is_train = (o["split"] == "train").to_numpy()
    is_test  = (o["split"] == "test").to_numpy()
    lineage  = o["lineage"].to_numpy()
    apoe     = o["apoe_carrier"].to_numpy()
    for lin in ("microglia", "astrocyte"):
        in_lin = (lineage == lin) & np.isin(apoe, list(APOE_KEEP))
        tr, te = is_train & in_lin, is_test & in_lin
        y = (apoe == "carrier").astype(int)
        print(f"\n[{tier}/{lin}] train {int(tr.sum())} / test {int(te.sum())} cells")
        for layer in LAYERS[tier]:
            mets = {m: apoe_metrics(EMB[(tier, m, layer)], tr, te, y) for m in MODELS}
            k_zs, s_zs = mets["zeroshot"]
            print(f"  {layer}: zero-shot k-NN {k_zs:.4f} | sil {s_zs:+.4f}")
            for m in MODELS[1:]:
                k_m, s_m = mets[m]
                dk, ds = (k_m - k_zs) * 100, s_m - s_zs
                kv = "underpowered" if POWER_2[tier][lin] else band_knn(dk)
                sv = "underpowered" if POWER_2[tier][lin] else band_sil(ds)
                EVAL2[(tier, m, lin, layer)] = {
                    "knn_bacc_zeroshot": round(float(k_zs), 4), "knn_bacc_cpt": round(float(k_m), 4),
                    "knn_delta_pp": round(float(dk), 2), "knn_verdict": kv,
                    "sil_zeroshot": round(float(s_zs), 4), "sil_cpt": round(float(s_m), 4),
                    "sil_delta": round(float(ds), 4), "sil_verdict": sv,
                    "sil_role": "corroborating_only"}
                print(f"    {m:12s} k-NN {k_m:.4f} (Δ{dk:+6.2f} pp [{kv}]) | "
                      f"sil {s_m:+.4f} (Δ{ds:+.4f} [{sv}])")

# --- non-circular reproducibility check against colab_12's recorded aggregated result ---
if SMOKE:
    print("\n[SMOKE] skipping the colab_12 cross-check (subsampled cells, numbers not comparable).")
else:
    ref = audit["geneformer_cpt_evals"]["eval2_apoe_recovery"]
    print("\n=== reproducibility vs colab_12 (aggregated, L-1, full population) ===")
    REPRO_HARD, REPRO_WARN = 0.01, 0.005
    for lin in ("microglia", "astrocyte"):
        k = f"{lin}|L-1"
        assert k in ref, f"colab_12 recorded no eval2 entry for {k}"
        for tag, key in (("zero-shot", "knn_bacc_zeroshot"), ("aggregated", "knn_bacc_cpt")):
            got, exp = EVAL2[("full", "aggregated", lin, "L-1")][key], ref[k][key]
            d = abs(got - exp)
            print(f"  {lin:10s} {tag:10s}: colab_15 k-NN {got:.4f} vs colab_12 {exp:.4f}  (|Δ| {d:.4f})")
            assert d <= REPRO_HARD, (
                f"{lin} {tag}: k-NN balanced accuracy {got:.4f} differs from colab_12's recorded "
                f"{exp:.4f} by {d:.4f} -- something in the substrate/split/alignment moved")
            if d > REPRO_WARN:
                print(f"    WARNING: {d:.4f} exceeds a version-drift-sized difference.")


[full/microglia] train 30499 / test 9108 cells
  L-1: zero-shot k-NN 0.4575 | sil +0.0614
    aggregated   k-NN 0.4587 (Δ +0.12 pp [noise]) | sil +0.0618 (Δ+0.0004 [noise])
    seaad        k-NN 0.4592 (Δ +0.17 pp [noise]) | sil +0.0549 (Δ-0.0064 [regression])
    li2025       k-NN 0.4603 (Δ +0.28 pp [noise]) | sil +0.0582 (Δ-0.0032 [regression])
    haney2024    k-NN 0.4561 (Δ -0.14 pp [noise]) | sil +0.0573 (Δ-0.0040 [regression])

[full/astrocyte] train 49620 / test 13448 cells
  L-1: zero-shot k-NN 0.4020 | sil +0.0484
    aggregated   k-NN 0.4019 (Δ -0.01 pp [noise]) | sil +0.0479 (Δ-0.0005 [regression])
    seaad        k-NN 0.3985 (Δ -0.35 pp [noise]) | sil +0.0411 (Δ-0.0073 [regression])
    li2025       k-NN 0.4031 (Δ +0.12 pp [noise]) | sil +0.0468 (Δ-0.0016 [regression])
    haney2024    k-NN 0.4037 (Δ +0.17 pp [noise]) | sil +0.0456 (Δ-0.0028 [regression])

=== reproducibility vs colab_12 (aggregated, L-1, full population) ===
  microglia  zero-shot : colab_15 k-NN 0.4575 

> **Interpretation — APOE recovery is a floor null for every checkpoint, and the below-chance baseline is a separate, unresolved question (7b).** k-NN balanced accuracies for all five variants, both lineages, sit at or below the 0.50 chance line (0.3985-0.4603) — this is a floor null, not merely a flat one: every CPT-vs-zero-shot delta, in both directions, lands in the "noise" band, extending colab_12's aggregated-only floor null to all three per-study checkpoints. Silhouette deltas cross into the fixed "regression" label in 7 of these 8 combinations (all six per-study deltas, plus the aggregated/astrocyte one) — only aggregated/microglia stays "noise." Per the contract this is corroborating-only, not a verdict: the baselines themselves are already close to zero (+0.048 to +0.061), so these are small movements around an already near-null starting point, and k-NN (flat throughout) is what decides. One thing worth being precise about, connecting to 7a's new confound table: the below-chance k-NN baselines are NOT clean evidence that the confound documented in 7a isn't leaking through. The naive expectation is that a study-driven shortcut would push accuracy above chance, but that only holds if the study→carrier association has the same sign among the donors a classifier trains on and the donors it's tested on; 7a shows that sign actually differs by study (Li2025 and Haney2024 skew carrier, SEA-AD skews noncarrier), so a study-following classifier could just as easily land below chance if the train/test donor split shifts which studies dominate which side. This run cannot distinguish "no APOE signal at all" from "a confound-driven signal whose sign this donor split happens to invert" — that would need the same study×apoe_carrier breakdown 7a runs on the held-out set, run on the train-split donors too, which this notebook doesn't do. What is solid regardless of which explanation is right: every CPT-vs-zero-shot k-NN delta is null, since each CPT checkpoint is compared against the same zero-shot baseline under the same unresolved confound. The reproducibility check again matches colab_12's stored zero-shot and aggregated k-NN values exactly (`|Δ| 0.0000`, both lineages).

## 8 — Cross-regime read: per-study vs aggregated, and drift vs eval

### 8a — Primary comparison: all five variants on the same global held-out cells

This is the primary comparison the locked design specifies — every variant scored on the **same** global held-out population, so a difference between rows is a difference between models and not between the cells they were measured on. That framing is the direct lesson of colab_14: its first-reported "flat per-study drift" turned out to be an artifact of measuring each checkpoint on its own study's cells, and the flatness disappeared once the same population was used for all three.

Two things are printed. First, per eval and lineage, each checkpoint's delta side by side, with the best per-study checkpoint compared against the aggregated one — the "does any per-study checkpoint move an axis the aggregated one did not" question. Second, each checkpoint's `drift_all` next to the magnitude of its eval deltas, because colab_14 established a 1.81x spread in drift (SEA-AD > Haney2024 > Li2025) which predicts an ordering if drift and decodable structure are related at all.

Read the ordering column as descriptive only. Three checkpoints cannot support a correlation, no test is run, and the null-result context makes this weaker still: if every delta is inside the noise band then their ordering is an ordering of noise, and the honest conclusion is that drift magnitude has no demonstrated eval consequence — not that it has a weak one.

In [15]:
PER_STUDY_MODELS = [SLUG[s] for s in STUDIES]

if EXTRACT_L0:
    print("NOTE: 'full' and 'sub' rows below are DIFFERENT cell populations (full population vs. "
          "the donor-stratified subsample) with different probe training sets -- compare within a "
          "tier, not across tiers. The -1->L0 contrast (both rows measured on 'sub') is the one "
          "within-tier comparison this design supports; 'full/L-1' vs 'sub/L-1' is not.")

print("=== EVAL #1 (substate probe) -- Δ balanced accuracy vs zero-shot, pp ===")
for tier in TIERS:
    for layer in LAYERS[tier]:
        print(f"\n[{tier} / {layer}]")
        print(f"  {'lineage':11s} {'aggregated':>12s} " + " ".join(f"{m:>12s}" for m in PER_STUDY_MODELS)
              + "   best per-study vs aggregated")
        for lin in BINARY:
            row = {m: EVAL1[(tier, m, lin, layer)] for m in MODELS[1:]}
            best = max(PER_STUDY_MODELS, key=lambda m: row[m]["delta_pp"])
            gap  = row[best]["delta_pp"] - row["aggregated"]["delta_pp"]
            print(f"  {lin:11s} {row['aggregated']['delta_pp']:>+12.2f} "
                  + " ".join(f"{row[m]['delta_pp']:>+12.2f}" for m in PER_STUDY_MODELS)
                  + f"   {best} {gap:+.2f} pp")
            vs = {m: row[m]["verdict"] for m in MODELS[1:]}
            print(f"  {'':11s} verdicts: {vs}")

print("\n=== EVAL #2 (APOE k-NN) -- Δ balanced accuracy vs zero-shot, pp ===")
for tier in TIERS:
    for layer in LAYERS[tier]:
        print(f"\n[{tier} / {layer}]")
        for lin in ("microglia", "astrocyte"):
            row  = {m: EVAL2[(tier, m, lin, layer)] for m in MODELS[1:]}
            base = row["aggregated"]["knn_bacc_zeroshot"]
            floor_note = ("   <-- at/below the 0.50 chance line: a floor null, deltas say little"
                          if base <= 0.52 else "")
            print(f"  {lin:11s} zero-shot k-NN bacc {base:.4f}{floor_note}")
            deltas = "  ".join(f"{m} {row[m]['knn_delta_pp']:>+7.2f} pp" for m in MODELS[1:])
            print(f"    {deltas}")
            verdicts = {m: row[m]["knn_verdict"] for m in MODELS[1:]}
            print(f"    verdicts: {verdicts}")

# --- drift vs |eval delta|: descriptive ordering only, n=3 checkpoints ---
print("\n=== detector #1 drift vs eval-delta magnitude (DESCRIPTIVE, n=3 -- no test) ===")
_ref_tier, _ref_layer = "full", "L-1"
rows = []
for m in PER_STUDY_MODELS + ["aggregated"]:
    d1 = [abs(EVAL1[(_ref_tier, m, lin, _ref_layer)]["delta_pp"]) for lin in BINARY]
    d2 = [abs(EVAL2[(_ref_tier, m, lin, _ref_layer)]["knn_delta_pp"]) for lin in ("microglia", "astrocyte")]
    rows.append({"model": m, "drift_all": round(DRIFT_ALL[m], 5),
                 "drift_global_test": round(DRIFT_GLOBAL_TEST[m], 5), "steps": STEPS[m],
                 "mean_abs_eval1_pp": round(float(np.mean(d1)), 2),
                 "mean_abs_eval2knn_pp": round(float(np.mean(d2)), 2)})
drift_tbl = pd.DataFrame(rows).sort_values("drift_all", ascending=False)
print(drift_tbl.to_string(index=False))

_ps = drift_tbl[drift_tbl["model"].isin(PER_STUDY_MODELS)]
print(f"\n  per-study drift_all order (high->low, full population):         {list(_ps['model'])}")
_ps_test_order = list(_ps.sort_values("drift_global_test", ascending=False)["model"])
print(f"  per-study drift_global_test order (high->low, eval population): {_ps_test_order}")
if list(_ps["model"]) != _ps_test_order:
    print("  ORDER DOES NOT SURVIVE moving from the full population to the eval-scored population -- "
          "drift_all's ordering is NOT a population-matched prediction for the eval deltas below "
          "(the colab_14 lesson: drift is population-sensitive, not just checkpoint-sensitive).")
print(f"  by mean |eval#1 Δ|:    {list(_ps.sort_values('mean_abs_eval1_pp', ascending=False)['model'])}")
print(f"  by mean |eval#2 kNN Δ|:{list(_ps.sort_values('mean_abs_eval2knn_pp', ascending=False)['model'])}")
_all_noise = all(EVAL1[(_ref_tier, m, lin, _ref_layer)]["verdict"] in ("noise", "underpowered")
                 for m in PER_STUDY_MODELS for lin in BINARY) and \
             all(EVAL2[(_ref_tier, m, lin, _ref_layer)]["knn_verdict"] in ("noise", "underpowered")
                 for m in PER_STUDY_MODELS for lin in ("microglia", "astrocyte"))
if _all_noise:
    print("\n  every per-study eval delta is inside the noise band (or underpowered): the orderings "
          "above are orderings of noise. The supported statement is that drift magnitude has NO "
          "demonstrated eval consequence -- not that it has a weak one.")

=== EVAL #1 (substate probe) -- Δ balanced accuracy vs zero-shot, pp ===

[full / L-1]
  lineage       aggregated        seaad       li2025    haney2024   best per-study vs aggregated
  microglia          -0.12        -0.10        -0.13        +0.10   haney2024 +0.22 pp
              verdicts: {'aggregated': 'noise', 'seaad': 'noise', 'li2025': 'noise', 'haney2024': 'noise'}
  astrocyte          -0.17        +0.11        -0.15        +0.01   seaad +0.28 pp
              verdicts: {'aggregated': 'noise', 'seaad': 'noise', 'li2025': 'noise', 'haney2024': 'noise'}

=== EVAL #2 (APOE k-NN) -- Δ balanced accuracy vs zero-shot, pp ===

[full / L-1]
  microglia   zero-shot k-NN bacc 0.4575   <-- at/below the 0.50 chance line: a floor null, deltas say little
    aggregated   +0.12 pp  seaad   +0.17 pp  li2025   +0.28 pp  haney2024   -0.14 pp
    verdicts: {'aggregated': 'noise', 'seaad': 'noise', 'li2025': 'noise', 'haney2024': 'noise'}
  astrocyte   zero-shot k-NN bacc 0.4020   <-- at/below t

> **Interpretation — every verdict is noise, and drift magnitude shows no demonstrated eval consequence (8a).** This is the primary cross-regime comparison the design calls for: all four CPT checkpoints, both evals, both lineages, scored on the identical held-out cells. Every single verdict comes back "noise" — no per-study checkpoint moves either axis meaningfully, and none beats the aggregated checkpoint's own (also null) result by more than a fraction of a percentage point. The drift-vs-eval-delta table is the more interesting part to read carefully. The per-study drift ordering (SEA-AD > Haney2024 > Li2025) is identical whether measured on the full 142,588-cell population or on just the eval-scored global test split — a stability check that passed. But the eval-delta orderings do not track that drift ordering at all: ranked by mean absolute eval#1 delta the order is Li2025 > SEA-AD > Haney2024, and ranked by mean absolute eval#2 k-NN delta it's SEA-AD > Li2025 > Haney2024 — neither matches the drift order, nor each other. The cell's own printed caveat states the correct reading of this: because every per-study eval delta lies inside the noise band, these orderings are orderings of noise, not a weak trend — ranking three noise values always produces *some* ranking, and treating that ranking as evidence would be the specific mistake this print exists to block. The supported conclusion is that drift magnitude has no demonstrated eval consequence in this data, not that the relationship is weak.

### 8b — Supplementary: each checkpoint on its **own** study's held-out cells

The locked design names an in-domain slice as a supplementary table alongside the global primary, and it is the natural question for a per-study checkpoint: does a model trained on one study help on that study's own held-out donors, even if it does nothing globally? The probe's training set is held identical to 8a (pooled train-split cells) and only the *test* population is sliced, so the change from 8a is one variable, not two.

**These slices are not comparable to each other.** Each row is measured on a different set of cells, which is precisely the trap colab_14's `drift_test` fell into — three numbers that looked flat only because each described a different population. Compare a per-study checkpoint against the *aggregated* checkpoint **within** a row, never one row against another. The slices are also much thinner than the global set (see §3a's printed test-study fractions), so the power gate will legitimately mark some of them unevaluable — Haney2024's especially, being the smallest study.

In [16]:
EVAL_INDOMAIN = {}
tier = "full"
o = OBS[tier]
is_train = (o["split"] == "train").to_numpy()
is_test  = (o["split"] == "test").to_numpy()
lineage  = o["lineage"].to_numpy()
substate = o["substate"].to_numpy()
apoe     = o["apoe_carrier"].to_numpy()
study    = o["study_id"].to_numpy()

# zeroshot/aggregated are fit ONCE per lineage here rather than once per (lineage, study) below --
# their training set (`tr`) is the pooled train split, identical across the 3 in-domain slices;
# only the held-out `te` slice differs per study, and that only affects prediction, not fitting.
# The "own" model is genuinely different per study (a different checkpoint's embedding), so it is
# still fit fresh inside the loop -- there is no waste to remove there.
def _fit_probe(X, tr, y):
    s = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(s.transform(X[tr]), y[tr])
    return s, clf

def _fit_knn(X, tr, y):
    s = StandardScaler().fit(X[tr])
    knn = KNeighborsClassifier(n_neighbors=15).fit(s.transform(X[tr]), y[tr])
    return s, knn

_eval1_fit, _eval2_fit = {}, {}
for lin, (neg, pos) in BINARY.items():
    is_bin = np.isin(substate, [neg, pos])
    tr = is_train & (lineage == lin) & is_bin
    y  = (substate == pos).astype(int)
    for m in ("zeroshot", "aggregated"):
        _eval1_fit[(lin, m)] = _fit_probe(EMB[(tier, m, "L-1")], tr, y)
for lin in ("microglia", "astrocyte"):
    keep = np.isin(apoe, list(APOE_KEEP))
    tr = is_train & (lineage == lin) & keep
    y  = (apoe == "carrier").astype(int)
    for m in ("zeroshot", "aggregated"):
        _eval2_fit[(lin, m)] = _fit_knn(EMB[(tier, m, "L-1")], tr, y)

print("=== in-domain slices (SUPPLEMENTARY; rows are DIFFERENT populations -- do not compare across) ===")
for st in STUDIES:
    slug = SLUG[st]
    in_st = study == st
    o_st  = o[is_test & in_st]
    print(f"\n[{st}] held-out cells in this study: {int((is_test & in_st).sum())} "
          f"({o_st['donor_id'].nunique()} donors)")
    for lin, (neg, pos) in BINARY.items():
        is_bin = np.isin(substate, [neg, pos])
        tr = is_train & (lineage == lin) & is_bin                    # pooled train, identical to 8a
        te = is_test  & (lineage == lin) & is_bin & in_st            # in-domain test slice only
        y  = (substate == pos).astype(int)
        n_d = o.loc[te, "donor_id"].nunique()
        per_cls = {c: int(((substate == c) & te).sum()) for c in (neg, pos)}
        thin = (n_d < MIN_TEST_DONORS) or any(v < MIN_TEST_CELLS for v in per_cls.values()) \
               or len(np.unique(y[te])) < 2
        if thin:
            print(f"  eval1 {lin:10s}: UNEVALUABLE -- {int(te.sum())} cells, {n_d} donors, {per_cls}")
            EVAL_INDOMAIN[(slug, "eval1", lin)] = {"verdict": "underpowered",
                                                   "n_test": int(te.sum()), "n_donors": int(n_d)}
            continue
        s_zs, clf_zs   = _eval1_fit[(lin, "zeroshot")]
        s_agg, clf_agg = _eval1_fit[(lin, "aggregated")]
        b_zs  = balanced_accuracy_score(y[te], clf_zs.predict(s_zs.transform(EMB[(tier, "zeroshot", "L-1")][te])))
        b_agg = balanced_accuracy_score(y[te], clf_agg.predict(s_agg.transform(EMB[(tier, "aggregated", "L-1")][te])))
        b_cpt = probe_bacc(EMB[(tier, slug, "L-1")], tr, te, y)   # own-study model differs every iteration
        d_cpt, d_agg = (b_cpt - b_zs) * 100, (b_agg - b_zs) * 100
        EVAL_INDOMAIN[(slug, "eval1", lin)] = {
            "bacc_zeroshot": round(float(b_zs), 4), "bacc_own": round(float(b_cpt), 4),
            "bacc_aggregated": round(float(b_agg), 4), "delta_own_pp": round(float(d_cpt), 2),
            "delta_aggregated_pp": round(float(d_agg), 2), "verdict": band_probe(d_cpt),
            "n_test": int(te.sum()), "n_donors": int(n_d)}
        print(f"  eval1 {lin:10s}: zs {b_zs:.4f} | own {b_cpt:.4f} (Δ{d_cpt:+.2f} pp "
              f"[{band_probe(d_cpt)}]) | aggregated {b_agg:.4f} (Δ{d_agg:+.2f} pp) "
              f"| {int(te.sum())} cells / {n_d} donors")

    for lin in ("microglia", "astrocyte"):
        keep = np.isin(apoe, list(APOE_KEEP))
        tr = is_train & (lineage == lin) & keep
        te = is_test  & (lineage == lin) & keep & in_st
        y  = (apoe == "carrier").astype(int)
        n_d = o.loc[te, "donor_id"].nunique()
        per_cls = {c: int(((apoe == c) & te).sum()) for c in sorted(APOE_KEEP)}
        thin = (n_d < MIN_TEST_DONORS) or any(v < MIN_TEST_CELLS for v in per_cls.values()) \
               or len(np.unique(y[te])) < 2
        if thin:
            print(f"  eval2 {lin:10s}: UNEVALUABLE -- {int(te.sum())} cells, {n_d} donors, {per_cls}")
            EVAL_INDOMAIN[(slug, "eval2", lin)] = {"verdict": "underpowered",
                                                   "n_test": int(te.sum()), "n_donors": int(n_d)}
            continue
        s_zs, knn_zs   = _eval2_fit[(lin, "zeroshot")]
        s_agg, knn_agg = _eval2_fit[(lin, "aggregated")]
        k_zs  = balanced_accuracy_score(y[te], knn_zs.predict(s_zs.transform(EMB[(tier, "zeroshot", "L-1")][te])))
        k_agg = balanced_accuracy_score(y[te], knn_agg.predict(s_agg.transform(EMB[(tier, "aggregated", "L-1")][te])))
        k_cpt, _ = apoe_metrics(EMB[(tier, slug, "L-1")], tr, te, y)   # own-study model differs every iteration
        d_cpt, d_agg = (k_cpt - k_zs) * 100, (k_agg - k_zs) * 100
        EVAL_INDOMAIN[(slug, "eval2", lin)] = {
            "knn_zeroshot": round(float(k_zs), 4), "knn_own": round(float(k_cpt), 4),
            "knn_aggregated": round(float(k_agg), 4), "delta_own_pp": round(float(d_cpt), 2),
            "delta_aggregated_pp": round(float(d_agg), 2), "verdict": band_knn(d_cpt),
            "n_test": int(te.sum()), "n_donors": int(n_d)}
        print(f"  eval2 {lin:10s}: zs {k_zs:.4f} | own {k_cpt:.4f} (Δ{d_cpt:+.2f} pp "
              f"[{band_knn(d_cpt)}]) | aggregated {k_agg:.4f} (Δ{d_agg:+.2f} pp) "
              f"| {int(te.sum())} cells / {n_d} donors")

=== in-domain slices (SUPPLEMENTARY; rows are DIFFERENT populations -- do not compare across) ===

[SEA-AD] held-out cells in this study: 13505 (10 donors)
  eval1 microglia : zs 0.9130 | own 0.9107 (Δ-0.24 pp [noise]) | aggregated 0.9132 (Δ+0.01 pp) | 4232 cells / 10 donors
  eval1 astrocyte : zs 0.7951 | own 0.7961 (Δ+0.10 pp [noise]) | aggregated 0.7928 (Δ-0.24 pp) | 5601 cells / 10 donors
  eval2 microglia : zs 0.4473 | own 0.4536 (Δ+0.62 pp [noise]) | aggregated 0.4523 (Δ+0.50 pp) | 4842 cells / 9 donors
  eval2 astrocyte : zs 0.4595 | own 0.4578 (Δ-0.17 pp [noise]) | aggregated 0.4587 (Δ-0.08 pp) | 7418 cells / 9 donors

[Li2025] held-out cells in this study: 7136 (8 donors)
  eval1 microglia : zs 0.8535 | own 0.8532 (Δ-0.03 pp [noise]) | aggregated 0.8572 (Δ+0.37 pp) | 2409 cells / 8 donors
  eval1 astrocyte : zs 0.8615 | own 0.8601 (Δ-0.14 pp [noise]) | aggregated 0.8563 (Δ-0.52 pp) | 3304 cells / 8 donors
  eval2 microglia : zs 0.5079 | own 0.5126 (Δ+0.47 pp [noise]) | aggrega

> **Interpretation — supplementary in-domain slices, all noise, thin donor counts as expected (8b).** Each study's own checkpoint is scored here on that same study's own held-out donors (SEA-AD 13,505 cells/10 donors, Li2025 7,136 cells/8 donors, Haney2024 3,160 cells/4 donors) — explicitly not comparable row-to-row, since each row is a different cell population, only comparable within a row against the aggregated checkpoint's score on the same slice. Every delta shown, in both directions, is small (max magnitude about 1.07pp, Haney2024 astrocyte eval#2) and consistent with the same overall null seen in the primary comparison. The donor counts here are visibly thinner than the pooled global test set (as low as 4 donors for Haney2024) — this is exactly why this table is supplementary rather than primary, not a defect: a per-study in-domain slice necessarily has far fewer held-out donors than the pooled cross-study test set, since it only draws from the one study's own donor pool.

## 9 — Summary and handoff

### 9a — Verdict table, append the audit trace, print the commit commands

Everything assembled into one table and written to `outputs/audit_report.json` under `geneformer_cpt_per_study_evals`, alongside the tier/extraction-point configuration, the power floors actually applied, and the embedding files each number came from. The `tier_2_run` field records whether the L0 contrast exists at all, so a later reader cannot mistake a tier-1-only run for a dual-extraction one.

A `SMOKE` run does **not** write the trace — its numbers are plumbing, not results. Commit commands are printed for the WSL side, since Colab has no git credentials.

In [17]:
import shlex

def _k(*parts):
    return "|".join(str(p) for p in parts)

print("=== EVAL #1 (substate probe) ===")
for (tier, m, lin, layer), r in EVAL1.items():
    print(f"  {tier:5s} {m:12s} {lin:10s} {layer:4s}: {r['bacc_zeroshot']:.4f} -> {r['bacc_cpt']:.4f}"
          f"  Δ{r['delta_pp']:+6.2f} pp  [{r['verdict']}]")
print("\n=== EVAL #2 (APOE recovery) ===")
for (tier, m, lin, layer), r in EVAL2.items():
    print(f"  {tier:5s} {m:12s} {lin:10s} {layer:4s}: k-NN Δ{r['knn_delta_pp']:+6.2f} pp "
          f"[{r['knn_verdict']}] | sil Δ{r['sil_delta']:+.4f} [{r['sil_verdict']}]")
print("\n=== in-domain supplementary (not comparable across studies) ===")
for (slug, ev, lin), r in EVAL_INDOMAIN.items():
    print(f"  {slug:12s} {ev} {lin:10s}: {r['verdict']:12s} "
          f"({r['n_test']} cells / {r['n_donors']} donors)")

if SMOKE:
    print("\n[SMOKE] audit trace NOT written (plumbing run).")
else:
    with open(AUDIT_PATH) as f:
        report = json.load(f)
    report["geneformer_cpt_per_study_evals"] = {
        "status": "computed", "date": TODAY, "regime": "per_study", "seed": SEED,
        "reads_run": "geneformer_cpt_per_study",
        "reference_runs": ["geneformer_zeroshot", "geneformer_cpt_aggregated_v2"],
        "geneformer_commit": GENEFORMER_COMMIT,
        "tier_2_run": bool(EXTRACT_L0),
        "tiers": {t: {"layers": LAYERS[t], "n_cells": int(len(OBS[t]))} for t in TIERS},
        "extraction_points": {"L-1": "second-to-last (pipeline readout)",
                              "L0": "last layer (captures layer 11's absorbed share only; the "
                                    "head is unreachable as an embedding)"},
        "power_floors": {"min_test_donors": MIN_TEST_DONORS, "min_test_cells": MIN_TEST_CELLS,
                         "note": "min_test_cells is new in colab_15 -- a plausibility floor against "
                                 "single-donor-dominated / thin classes, not a calibrated SE bound "
                                 "(see 4b); needs sign-off"},
        "detector_2_gate": {
            "status": "not_run_for_per_study",
            "any_verdict_pending_gate": bool(
                any(r["verdict"] in ("meaningful", "decisive") for r in EVAL1.values()) or
                any(r["knn_verdict"] in ("meaningful", "decisive") for r in EVAL2.values())),
            "note": ("Detector #2 (forgetting, colab_13) has only ever been run for the aggregated "
                     "checkpoint. docs/EVALUATION_CONTRACT.md requires clearing detector #2 in "
                     "addition to the meaningful band before any regime's result is read as a win. "
                     "Any 'meaningful'/'decisive' verdict recorded here is UNGATED until a per-study "
                     "forgetting probe exists -- read as a candidate win pending that gate, not a "
                     "confirmed one.")},
        "sub_cap": SUB_CAP if EXTRACT_L0 else None,
        "detector_1_drift_all": {m: DRIFT_ALL[m] for m in MODELS[1:]},
        "training_steps": {m: STEPS[m] for m in MODELS[1:]},
        "eval1_substate_probe": {_k(t, m, lin, l): r for (t, m, lin, l), r in EVAL1.items()},
        "eval2_apoe_recovery":  {_k(t, m, lin, l): r for (t, m, lin, l), r in EVAL2.items()},
        "eval_in_domain_supplementary": {_k(s, e, lin): r for (s, e, lin), r in EVAL_INDOMAIN.items()},
        "embedding_files": {
            "L-1": {m: os.path.relpath(L1_PATHS[m], DRIVE_ROOT) for m in MODELS},
            "L0":  ({m: os.path.relpath(L0_PATHS[m], DRIVE_ROOT) for m in MODELS}
                    if EXTRACT_L0 else None)},
        "note": ("Primary = all variants on the same global held-out cells (population-matched, the "
                 "colab_14 drift_test lesson). In-domain slices are supplementary and NOT comparable "
                 "across studies. Eval #2: k-NN load-bearing, silhouette corroborating-only."),
    }
    with open(AUDIT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("\naudit trace appended ->", AUDIT_PATH)
    rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_PATH)]
    print("\n=== Commit + push (from WSL) ===")
    print("  git add " + " ".join(shlex.quote(r) for r in rel))
    print("  git commit -m 'colab_15: Geneformer CPT evals #1 + #2 on the three per-study checkpoints'")
    print("  git push")

=== EVAL #1 (substate probe) ===
  full  aggregated   microglia  L-1 : 0.9069 -> 0.9057  Δ -0.12 pp  [noise]
  full  seaad        microglia  L-1 : 0.9069 -> 0.9058  Δ -0.10 pp  [noise]
  full  li2025       microglia  L-1 : 0.9069 -> 0.9056  Δ -0.13 pp  [noise]
  full  haney2024    microglia  L-1 : 0.9069 -> 0.9079  Δ +0.10 pp  [noise]
  full  aggregated   astrocyte  L-1 : 0.7784 -> 0.7767  Δ -0.17 pp  [noise]
  full  seaad        astrocyte  L-1 : 0.7784 -> 0.7795  Δ +0.11 pp  [noise]
  full  li2025       astrocyte  L-1 : 0.7784 -> 0.7769  Δ -0.15 pp  [noise]
  full  haney2024    astrocyte  L-1 : 0.7784 -> 0.7784  Δ +0.01 pp  [noise]

=== EVAL #2 (APOE recovery) ===
  full  aggregated   microglia  L-1 : k-NN Δ +0.12 pp [noise] | sil Δ+0.0004 [noise]
  full  seaad        microglia  L-1 : k-NN Δ +0.17 pp [noise] | sil Δ-0.0064 [regression]
  full  li2025       microglia  L-1 : k-NN Δ +0.28 pp [noise] | sil Δ-0.0032 [regression]
  full  haney2024    microglia  L-1 : k-NN Δ -0.14 pp [noise]

> **Interpretation — final verdict table, audit trace appended, no wins to gate (9a).** The consolidated table re-prints the same `EVAL1`/`EVAL2`/`EVAL_INDOMAIN` dictionaries that 6b, 7b, and 8b already populated earlier in this run's memory — a useful single-place summary, but not an independent check that "nothing drifted," since a dictionary re-printing itself cannot disagree with itself. Net result across the whole notebook: eval#1 (substate probe) and eval#2's k-NN (the load-bearing metric) are null for all three per-study CPT checkpoints, at the L−1 extraction point, in both lineages, on both the primary global comparison and the supplementary in-domain slices — extending colab_12's aggregated-only null finding to the full per-study regime. Eval#2's silhouette (corroborating-only) is not uniformly noise: 7 of its 8 combinations land in the fixed "regression" band (see 7b), though that doesn't change the k-NN-decided verdict. The audit entry appended under `geneformer_cpt_per_study_evals` records `tier_2_run: false` (so a later reader cannot mistake this tier-1-only run for a full dual-extraction one), and a `detector_2_gate` field noting that detector #2 (the forgetting probe) has only ever been run for the aggregated checkpoint. `any_verdict_pending_gate` is recorded as `false` here because the gate logic scans only `EVAL1`'s probe verdicts and `EVAL2`'s k-NN verdicts for "meaningful"/"decisive" — both are all-noise this run, so nothing trips it — but the same field does not scan silhouette verdicts or the in-domain (`EVAL_INDOMAIN`) table at all; that's harmless today only because every verdict everywhere happens to be noise or (for silhouette) capped at "regression," neither of which the gate is built to catch. A future run with a real in-domain or silhouette "win" would not automatically get flagged by this field. The printed git commands are for committing this run's artifacts (pip-freeze snapshot, env JSON, updated audit trail) from WSL, since Colab itself has no git credentials to push with — note the printed `git add` list does not include the executed notebook file itself, which also needs to be staged.

### Carried forward

- **What this closes.** The per-study regime now has the same eval battery the aggregated regime got in colab_12, scored on a population-matched global held-out set, so competency-spec item 4's "compare the three regimes" rests on eval outcomes rather than on drift alone.
- **What it does not close.** Eval #3 / detector #2 (forgetting) has been run only for the aggregated checkpoint (colab_13). Per the colab_12 ↔ colab_13 dependency, detector #2 gates any *win* recorded here — a per-study win would need its own forgetting probe before it can be read as a gain rather than as overtraining. This is now recorded machine-readably in the audit trace itself (`geneformer_cpt_per_study_evals.detector_2_gate`), not only in this prose — a downstream reader parsing only `audit_report.json` can no longer miss that any meaningful/decisive verdict here is ungated.
- **Still N=1 per regime.** One seed per checkpoint. Nothing here separates a regime effect from seed variation.
- **Open item needing a call.** `MIN_TEST_CELLS = 100` is a new power floor — a plausibility floor against single-donor-dominated / thin classes, not a calibrated SE bound (see 4b) — rather than inherited from the contract. If it is kept, `docs/EVALUATION_CONTRACT.md` should record it; if it is replaced, this notebook and colab_12's flagged gap both need the replacement.
- **Tier 2's L0 arm has no independent cross-check.** Unlike L-1 (checked against colab_12's recorded numbers above), colab_12's own L0 numbers are full-population, not subsample, so there is no reference to recompute the subsample's L0 embeddings against. Internal asserts (alignment, shape, per-cell tokenization invariance) are what stands behind it.
- **If tier 2 was not run**, every verdict here is blind to the head + layer-11 absorption the head-ablation arc identified, and any null should be stated with that limitation attached.